# FoundationPose Evaluation — Pipeline Completo

**Un solo notebook. Ejecutar celda por celda.**

Pipeline:
1. Setup entorno + GPU
2. Datasets BOP (desde Drive cache)
3. Instalar FoundationPose + nvdiffrast
4. Descargar pesos pre-entrenados
5. Inferencia YCB-V (5 escenas, checkpoints en Drive)
6. Inferencia T-LESS (5 escenas, checkpoints en Drive)
7. Metricas + Comparativa vs GDR-Net++
8. Figuras para Capitulo 6

**Checkpoints:** Si Colab se desconecta, re-ejecuta todo. El setup toma ~5 min y la inferencia retoma automaticamente.

**Requiere:** GPU T4 (Runtime > Change runtime type > T4 GPU)

## 0. Reset + Health Check

Esta celda hace 3 cosas:
1. **Health check**: reporta que hay en Drive (pesos, checkpoints, experimentos previos) para que sepas el estado antes de correr.
2. **Reset granular opcional**: elige que limpiar con `RESET_LEVEL`:
   - `'none'` (default): no borra nada. Retoma donde dejaste.
   - `'checkpoints'`: solo borra checkpoints de inferencia para empezar eval de cero (mantiene datasets, pesos).
   - `'build'`: borra FoundationPose clone + caches pip (si la compilacion quedo corrupta). Mantiene Drive intacto.
   - `'full'`: borra todo lo regenerable (clone, checkpoints, datasets extraidos en /content). **No toca pesos ni datasets_zips en Drive.**
3. **Nunca borra automaticamente**: pesos FP, zips de datasets, figuras ni JSONs de experimentos en Drive (tu trabajo). Si necesitas eso, hazlo manual.

In [ ]:
# ============================================================
# HEALTH CHECK + RESET granular
# ============================================================
# Configura RESET_LEVEL y ejecuta. Default 'none' solo reporta estado.
#
# Niveles:
#   'none'         : solo health check (no toca nada)
#   'checkpoints'  : borra fp_{ycbv,tless}_checkpoint.json en Drive
#   'build'        : borra /content/FoundationPose y cache pip (recompila)
#   'full'         : checkpoints + build + /content/datasets (re-baja)
#
# NUNCA borra: pesos, datasets_zips, figures, experiments en Drive.
RESET_LEVEL = 'checkpoints'  # limpia ckpt viejos (pre-v2) antes de una corrida nueva

import os, shutil, subprocess
from pathlib import Path
from google.colab import drive

# --- Mount Drive (idempotente) ---
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

DRIVE_ROOT      = '/content/drive/MyDrive/TFM'
DRIVE_WEIGHTS   = f'{DRIVE_ROOT}/weights/foundationpose'
DRIVE_ZIPS      = f'{DRIVE_ROOT}/datasets_zips'
DRIVE_CKPTS     = f'{DRIVE_ROOT}/checkpoints'
DRIVE_EXP       = f'{DRIVE_ROOT}/experiments/foundationpose_eval'
DRIVE_FIG       = f'{DRIVE_ROOT}/figures/chapter6'

def _size_mb(path):
    if not os.path.exists(path): return 0
    if os.path.isfile(path): return os.path.getsize(path) / 1e6
    total = 0
    for r, _, fs in os.walk(path):
        for f in fs:
            try: total += os.path.getsize(os.path.join(r, f))
            except OSError: pass
    return total / 1e6

def _gb_free(p):
    try: return shutil.disk_usage(p).free / 1e9
    except Exception: return 0.0

print('=' * 60)
print('HEALTH CHECK — Drive TFM')
print('=' * 60)
print(f'  Drive libre: {_gb_free("/content/drive/MyDrive/"):.1f} GB')
print(f'  /content libre: {_gb_free("/content/"):.1f} GB')
print()

# Pesos FP (scorer esperado ~181 MB, refiner ~65 MB)
scorer_mb = _size_mb(f'{DRIVE_WEIGHTS}/2024-01-11-20-02-45')
refiner_mb = _size_mb(f'{DRIVE_WEIGHTS}/2023-10-28-18-33-37')
w_scorer_ok = scorer_mb >= 150  # ~181 MB real
w_refiner_ok = refiner_mb >= 50  # ~65 MB real
print(f'  Pesos FP Scorer:  {scorer_mb:6.1f} MB  {"OK" if w_scorer_ok else "FALTA/CORRUPTO"}')
print(f'  Pesos FP Refiner: {refiner_mb:6.1f} MB  {"OK" if w_refiner_ok else "FALTA/CORRUPTO"}')

# Datasets zips (6 archivos)
print(f'  Datasets zips en Drive:')
expected_zips = [
    ('ycbv_base.zip', 1),
    ('ycbv_models.zip', 300),
    ('ycbv_test_all.zip', 14000),
    ('tless_base.zip', 1),
    ('tless_models.zip', 20),
    ('tless_test_primesense_all.zip', 7000),
]
zips_ok = 0
for fn, min_mb in expected_zips:
    fp = f'{DRIVE_ZIPS}/{fn}'
    mb = _size_mb(fp)
    ok = mb >= min_mb
    if ok: zips_ok += 1
    print(f'    {fn:45s} {mb:7.0f} MB  {"OK" if ok else "FALTA/PEQUENO"}')
print(f'  → {zips_ok}/6 zips OK. Si < 6, celda 6 los re-descarga solo.')

# Checkpoints
print(f'  Checkpoints de inferencia:')
for fn in ['fp_ycbv_checkpoint.json', 'fp_tless_checkpoint.json']:
    fp = f'{DRIVE_CKPTS}/{fn}'
    if os.path.exists(fp):
        import json as _json
        try:
            c = _json.load(open(fp))
            n_obj = c.get('n_objects_evaluated', 0)
            n_sc = len(c.get('completed_scenes', []))
            print(f'    {fn}: {n_sc} escenas, {n_obj} objetos')
        except Exception:
            print(f'    {fn}: EXISTE pero ilegible')
    else:
        print(f'    {fn}: NO (empezara de cero)')

# Experimentos previos
if os.path.isdir(DRIVE_EXP):
    exps = sorted([f for f in os.listdir(DRIVE_EXP) if f.endswith('.json')])
    print(f'  Experimentos JSON previos: {len(exps)}')
    for e in exps[-3:]:
        print(f'    {e}')
if os.path.isdir(DRIVE_FIG):
    figs = sorted([f for f in os.listdir(DRIVE_FIG) if f.endswith('.png')])
    print(f'  Figuras previas: {len(figs)}')

print('=' * 60)

# ============================================================
# RESET granular
# ============================================================
if RESET_LEVEL == 'none':
    print('RESET_LEVEL=none. Nada que limpiar. Continua con las siguientes celdas.')
else:
    print(f'\nRESET_LEVEL={RESET_LEVEL!r}. Limpiando...')

    if RESET_LEVEL in ('checkpoints', 'full'):
        for fn in ['fp_ycbv_checkpoint.json', 'fp_tless_checkpoint.json']:
            fp = f'{DRIVE_CKPTS}/{fn}'
            if os.path.exists(fp):
                os.remove(fp)
                print(f'  [RESET] borrado {fp}')

    if RESET_LEVEL in ('build', 'full'):
        for t in ['/content/FoundationPose', '/content/repo_tfm/weights', '/root/.cache/pip']:
            if os.path.exists(t):
                shutil.rmtree(t, ignore_errors=True)
                print(f'  [RESET] borrado {t}')

    if RESET_LEVEL == 'full':
        # Borra datasets extraidos en /content (los zips de Drive se mantienen)
        for t in ['/content/datasets', '/content/zips_cache']:
            if os.path.exists(t):
                shutil.rmtree(t, ignore_errors=True)
                print(f'  [RESET] borrado {t}')

    print(f'[RESET] Hecho. Pon RESET_LEVEL=\'none\' y sigue.')
    print('NOTA: pesos FP, zips de datasets, figuras y experimentos en Drive NO se tocaron.')


## 1. Setup Entorno

In [ ]:
import torch
import os
import sys
import time

# --- GPU ---
assert torch.cuda.is_available(), "GPU requerida! Runtime > Change runtime type > T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# --- Repo ---
REPO_URL = "https://github.com/Giocrisrai/pose6dof-transformers-diffusion.git"
REPO_DIR = "/content/repo_tfm"

if not os.path.exists(REPO_DIR):
    print("Clonando repo...")
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Actualizando repo...")
    !cd {REPO_DIR} && git fetch origin main && git reset --hard origin/main

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
!git log --oneline -1

# --- Dependencias ---
!pip install -q uv 2>&1 | tail -1
!uv pip install --system -q trimesh diffusers accelerate transformers scikit-learn

# Restaurar numpy si fue downgradeado
import numpy as np
if int(np.__version__.split('.')[0]) < 2:
    print(f"numpy {np.__version__} < 2.0, restaurando...")
    !uv pip install --system -q "numpy>=2.0"
    import importlib; importlib.reload(np)

print(f"numpy {np.__version__} OK")

# Verificar imports del proyecto
from src.utils.metrics import add_metric, add_s_metric, compute_recall, compute_auc
from src.utils.dataset_loader import BOPDataset
print("Imports OK")

## 2. Google Drive + Datasets BOP

**Datasets:** YCB-Video + T-LESS desde el mirror oficial BOP en HuggingFace (~22 GB).

**Caching:**
- `USE_DRIVE_CACHE = True` (default): cachea zips en `Drive/TFM/datasets_zips/`. En re-runs, setup = ~10 s (vs 10-15 min re-descarga). Ideal si iteras mucho.
- `USE_DRIVE_CACHE = False`: baja zips a `/content/zips_cache/` (efimeros, se pierden al desconectar). Ideal si eres evaluador que solo corre 1 vez, o si tu Drive esta cerca de la quota.
- **Fallback automatico**: si tu Drive tiene menos de 30 GB libres, cambia solo a modo local, aunque `USE_DRIVE_CACHE=True`.
- **Integridad**: cada zip cacheado se valida con `unzip -t` antes de extraer. Si esta corrupto (caso clasico: Drive borro un archivo a medio subir), se purga y re-descarga automaticamente.
- **Idempotencia**: marker `.extracted_<zip>` por dataset evita re-extraer si ya existe el contenido.

**Si ves `ERROR: test/ no encontrado`**: la descarga o extraccion fallo. Revisa el mensaje anterior (WARN/DOWN). Opciones:
1. Reintenta la celda (idempotente, retoma desde donde fallo).
2. Fuerza modo local: `USE_DRIVE_CACHE = False`, reintenta.
3. Si persiste, borra `/content/datasets/` y `/content/zips_cache/` y reintenta.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil, subprocess, os, time

# Marker para verificar que este notebook es la version con thresholds reales.
# Si tu Colab imprime un valor distinto, cierra la pestana y reabre el enlace
# (File > Open notebook > GitHub > .../01_foundationpose_eval.ipynb).
_NOTEBOOK_VERSION = "zip-thresholds-bytes-aaa8bc4"
print(f"[VERSION] cell 6: {_NOTEBOOK_VERSION}")

# ============================================================
# Config de cache de datasets BOP (YCB-V + T-LESS, ~22 GB)
# ============================================================
# USE_DRIVE_CACHE=True (default): guarda zips en Drive para
#   re-runs rapidos (10 s vs 10-15 min si Colab se desconecta).
# USE_DRIVE_CACHE=False: baja zips a /content (efimero).
#   Util si tu Drive esta cerca de la quota, o si eres
#   evaluador y solo vas a correr 1 vez.
# Si Drive tiene menos de MIN_DRIVE_FREE_GB libres, hay
# fallback automatico a /content aunque USE_DRIVE_CACHE=True.
USE_DRIVE_CACHE = True
MIN_DRIVE_FREE_GB = 30  # 22 GB zips + margen

drive.mount("/content/drive")

DRIVE_ZIPS  = "/content/drive/MyDrive/TFM/datasets_zips"
LOCAL_ZIPS  = "/content/zips_cache"
LOCAL_DATA  = "/content/datasets"
WEIGHTS_DIR = "/content/drive/MyDrive/TFM/weights"

def _gb_free(path):
    try:
        return shutil.disk_usage(path).free / 1e9
    except Exception:
        return 0.0

drive_free   = _gb_free("/content/drive/MyDrive/")
content_free = _gb_free("/content/")

if USE_DRIVE_CACHE and drive_free >= MIN_DRIVE_FREE_GB:
    ZIPS_DIR, CACHE_MODE = DRIVE_ZIPS, "drive"
    print(f"[CACHE] Modo DRIVE ({drive_free:.0f} GB libres). Zips persistentes entre sesiones.")
elif USE_DRIVE_CACHE:
    ZIPS_DIR, CACHE_MODE = LOCAL_ZIPS, "local-fallback"
    print(f"[CACHE] Drive solo tiene {drive_free:.0f} GB libres (< {MIN_DRIVE_FREE_GB}). FALLBACK a /content (efimero).")
else:
    ZIPS_DIR, CACHE_MODE = LOCAL_ZIPS, "local"
    print(f"[CACHE] Modo LOCAL por config (USE_DRIVE_CACHE=False). Zips efimeros en {LOCAL_ZIPS} ({content_free:.0f} GB libres).")

for d in [ZIPS_DIR, LOCAL_DATA, WEIGHTS_DIR]:
    os.makedirs(d, exist_ok=True)

# Symlink: repo/data/datasets -> /content/datasets
REPO_DATA = f"{REPO_DIR}/data/datasets"
if os.path.islink(REPO_DATA):
    os.unlink(REPO_DATA)
elif os.path.isdir(REPO_DATA):
    shutil.rmtree(REPO_DATA)
os.makedirs(os.path.dirname(REPO_DATA), exist_ok=True)
os.symlink(LOCAL_DATA, REPO_DATA)
print(f"Symlink: {REPO_DATA} -> {LOCAL_DATA}")

# ============================================================
# Descarga robusta: verifica integridad y cae a local si Drive falla
# ============================================================
HF_BASE = "https://huggingface.co/datasets/bop-benchmark"

# Tamaños minimos esperados por archivo (bytes) — medidos con HEAD en HF.
# Un zip por debajo de esto se considera incompleto (Drive a veces deja
# stubs de subida fallida de 0-22 KB).
#
# Los _base.zip son muy pequenos (solo JSON de metadata). Los _models.zip
# son medianos (PLY de objetos). Los _test*.zip son enormes (imagenes).
_ZIP_MIN_BYTES = {
    "ycbv_base.zip":                      8_000,           # real 15_805 B
    "tless_base.zip":                    20_000,           # real 49_597 B
    "ycbv_models.zip":               400_000_000,          # real 524 MB
    "tless_models.zip":               25_000_000,          # real 33 MB
    "ycbv_test_all.zip":          12_000_000_000,          # real 14.9 GB
    "tless_test_primesense_all.zip": 6_000_000_000,        # real 8.3 GB
}

def _zip_ok(zip_path):
    """Valida (a) tamano sobre el minimo esperado y (b) integridad CRC.
    Detecta el caso classico de Drive dejando un .zip de 0 bytes."""
    if not os.path.exists(zip_path):
        return False
    name = os.path.basename(zip_path)
    min_bytes = _ZIP_MIN_BYTES.get(name, 1024)  # default 1 KB
    actual_bytes = os.path.getsize(zip_path)
    if actual_bytes < min_bytes:
        return False
    return subprocess.run(["unzip", "-tq", zip_path], capture_output=True).returncode == 0

def _wget(url, dst):
    """wget -c (reanuda). Devuelve True si el destino quedo como zip valido.
    Imprime diagnostico verboso si la validacion falla."""
    r = subprocess.run(["wget", "-q", "--show-progress", "-c", "-O", dst, url])
    if _zip_ok(dst):
        return True
    # Diagnostico detallado
    name = os.path.basename(dst)
    size = os.path.getsize(dst) if os.path.exists(dst) else 0
    min_b = _ZIP_MIN_BYTES.get(name, 1024)
    print(f"    [DIAG] wget exit={r.returncode}  size={size} B  min={min_b} B")
    if size > 0 and size >= min_b:
        # Paso size pero no CRC — imprimir primer error de unzip
        chk = subprocess.run(["unzip", "-tq", dst], capture_output=True, text=True)
        print(f"    [DIAG] unzip -tq exit={chk.returncode}  stderr={chk.stderr[:200]}")
    return False

def download_and_extract(dataset, filename, url):
    """Descarga zip (cache > re-download), verifica integridad, extrae con marker
    anti-re-extraccion. Si el cache esta corrupto lo purga y re-baja. Si Drive
    no puede escribir / descarga falla, fallback a /content."""
    extract_dir = f"{LOCAL_DATA}/{dataset}"
    os.makedirs(extract_dir, exist_ok=True)

    marker = f"{extract_dir}/.extracted_{filename}"
    zip_path = f"{ZIPS_DIR}/{filename}"

    # Validacion fuerte del marker: si existe pero el zip cacheado es invalido
    # (caso real: Drive dejo el zip a 0 bytes y aun asi el extract previo creo
    # marker porque unzip no fallo), borramos marker y zip para forzar re-bajada.
    if os.path.exists(marker):
        if os.path.exists(zip_path) and not _zip_ok(zip_path):
            print(f"  [INVALIDATE] marker existe pero {filename} es invalido. Re-bajando.")
            try: os.remove(marker)
            except Exception: pass
            try: os.remove(zip_path)
            except Exception: pass
        else:
            print(f"  [SKIP] {filename} ya extraido")
            return

    # Valida cache: si existe pero esta corrupto, borra para re-bajar.
    if os.path.exists(zip_path):
        if _zip_ok(zip_path):
            print(f"  [CACHE] {filename} ({os.path.getsize(zip_path)/1e6:.0f} MB) OK")
        else:
            print(f"  [WARN] {filename} cacheado esta corrupto/vacio, re-descargando")
            try: os.remove(zip_path)
            except Exception: pass

    # Descarga si falta. Si Drive falla, reintenta a /content.
    if not os.path.exists(zip_path):
        print(f"  [DOWN] {filename}  (destino: {CACHE_MODE})")
        if not _wget(url, zip_path):
            if zip_path.startswith(DRIVE_ZIPS):
                print(f"  [WARN] Escritura a Drive fallo o zip invalido. Fallback a /content.")
                try: os.remove(zip_path)
                except Exception: pass
                os.makedirs(LOCAL_ZIPS, exist_ok=True)
                zip_path = f"{LOCAL_ZIPS}/{filename}"
                if not _wget(url, zip_path):
                    raise RuntimeError(
                        f"Descarga fallida en Drive y en /content para {filename}. "
                        f"Revisa red o disponibilidad de {url}"
                    )
            else:
                raise RuntimeError(f"Descarga fallida para {filename}. URL: {url}")

    print(f"  [UNZIP] Extrayendo...")
    t0 = time.time()
    if subprocess.run(["unzip", "-qo", zip_path, "-d", extract_dir]).returncode != 0:
        # No toques el zip: puede ser recoverable con -F. Mejor fallar explicito.
        raise RuntimeError(
            f"unzip fallo para {filename} aun con integridad OK. "
            f"Borra {zip_path} y reintenta la celda."
        )
    Path(marker).touch()
    print(f"  [OK] {time.time()-t0:.0f}s")

def reorganize_bop(dataset_dir, nested_name, camera_src="camera.json"):
    """BOP-HF zips extraen con carpeta raiz duplicada (ycbv/ycbv/). Aplana."""
    nested = Path(dataset_dir) / nested_name
    if nested.exists():
        !cp -rn {nested}/* {dataset_dir}/ 2>/dev/null; true
    cam_src = Path(dataset_dir) / camera_src
    if cam_src.exists() and not (Path(dataset_dir) / "camera.json").exists():
        !cp {cam_src} {dataset_dir}/camera.json

def verify_dataset(dataset_dir, split):
    test_dir = Path(dataset_dir) / split
    if not test_dir.exists():
        print(f"  ERROR: {split}/ no encontrado en {dataset_dir}")
        return False
    scenes = sorted([d.name for d in test_dir.iterdir() if d.is_dir()])
    if not scenes:
        print(f"  ERROR: sin escenas en {split}/")
        return False
    rgb_dir = test_dir / scenes[0] / "rgb"
    imgs = sorted(rgb_dir.glob("*.*")) if rgb_dir.exists() else []
    print(f"  {len(scenes)} escenas, primera: {scenes[0]}, {len(imgs)} imgs ({imgs[0].name if imgs else 'N/A'})")
    return len(imgs) > 0

# ============================================================
# YCB-Video
# ============================================================
print("\n=== YCB-Video ===")
download_and_extract("ycbv", "ycbv_base.zip",     f"{HF_BASE}/ycbv/resolve/main/ycbv_base.zip")
download_and_extract("ycbv", "ycbv_models.zip",   f"{HF_BASE}/ycbv/resolve/main/ycbv_models.zip")
download_and_extract("ycbv", "ycbv_test_all.zip", f"{HF_BASE}/ycbv/resolve/main/ycbv_test_all.zip")

YCBV = f"{LOCAL_DATA}/ycbv"
reorganize_bop(YCBV, "ycbv", "camera_uw.json")
ok_ycbv = verify_dataset(YCBV, "test")

# ============================================================
# T-LESS
# ============================================================
print("\n=== T-LESS ===")
download_and_extract("tless", "tless_base.zip",                f"{HF_BASE}/tless/resolve/main/tless_base.zip")
download_and_extract("tless", "tless_models.zip",              f"{HF_BASE}/tless/resolve/main/tless_models.zip")
download_and_extract("tless", "tless_test_primesense_all.zip", f"{HF_BASE}/tless/resolve/main/tless_test_primesense_all.zip")

TLESS = f"{LOCAL_DATA}/tless"
reorganize_bop(TLESS, "tless", "camera_primesense.json")
!ln -sf models_cad {TLESS}/models 2>/dev/null; true
ok_tless = verify_dataset(TLESS, "test_primesense")

# ============================================================
# Resumen
# ============================================================
print("\n=== Disco ===")
!df -h /content | tail -1
print()
!du -sh {LOCAL_DATA}/*/ 2>/dev/null
print()
if CACHE_MODE.startswith("drive"):
    !du -sh {DRIVE_ZIPS}/ 2>/dev/null
else:
    !du -sh {LOCAL_ZIPS}/ 2>/dev/null
    print("[INFO] Zips en /content son efimeros: se borran al cerrar la sesion.")

if not (ok_ycbv and ok_tless):
    raise RuntimeError(
        "Datasets incompletos. No continues con las celdas de inferencia."
    )
print("\n[OK] Datasets listos.")

### Presupuesto de ejecucion (lee antes de correr)

**Colab T4 Free:** 15 GB VRAM, ~12 GB RAM, ~77 GB disk /content, ~12h de sesion, idle timeout 90 min.

**Esta notebook hace:**
- Clone FoundationPose + compilacion pytorch3d/nvdiffrast/mycpp: 15-30 min la primera vez.
- Datasets BOP (YCB-V test_all + T-LESS primesense): ~40 GB en /content.
- Inferencia: FoundationPose.register() es ~1-3 s por objeto en T4. Con los defaults (MAX_SCENES=5, MAX_IMAGES_PER_SCENE=50, ~3 objetos/imagen) = ~15-40 min por dataset.

**NO hagas esto:**
- Dos notebooks Colab con esta misma cuenta al mismo tiempo (conflicto Drive mount + doble quota GPU).
- Subir MAX_SCENES a todas las escenas sin checkpoint: full eval > 12 h, te quedas sin sesion.
- Cerrar la pestaña: Free = idle timeout mata la sesion. Usa Colab Pro+ si necesitas background.

**SI quieres paralelizar:**
- Ejecuta 01 (FoundationPose) en una cuenta Google y 02 (GDR-Net) en otra. Son independientes.
- Los checkpoints estan en Drive de cada cuenta, no chocan.

**Quota GPU T4 Free:** ~12 h/dia pero variable. Si te corta, espera 12-24 h. Considera Colab Pro ($10/mes) si vas a iterar mucho esta semana.


## 3. Instalar FoundationPose

In [ ]:
FP_DIR = "/content/FoundationPose"

if not os.path.exists(FP_DIR):
    print("Clonando FoundationPose...")
    !git clone --depth 1 https://github.com/NVlabs/FoundationPose.git {FP_DIR}
else:
    print(f"FoundationPose ya existe en {FP_DIR}")

# ============================================================
# PARCHE Utils.py: fallbacks para warp kernels (erode_depth, etc.)
# Warp no compila en Colab, pero estas funciones son esenciales.
# ============================================================
_utils_path = f"{FP_DIR}/Utils.py"
with open(_utils_path) as _f:
    _utils_content = _f.read()

if 'erode_depth_FALLBACK_INJECTED' not in _utils_content:
    print("[PATCH] Inyectando fallbacks warp en Utils.py...")
    _fallback_code = """

# ---- erode_depth_FALLBACK_INJECTED ----
# Fallback Python para funciones que usan NVIDIA Warp (no disponible en Colab)
import numpy as _fb_np

if 'erode_depth' not in dir():
    def erode_depth(depth, radius=2, depth_diff_thres=0.001,
                    ratio_thres=0.8, zfar=100, device='cuda'):
        import torch as _t
        is_tensor = isinstance(depth, _t.Tensor)
        d = depth.cpu().numpy() if is_tensor else _fb_np.array(depth)
        H, W = d.shape
        valid = (d >= 0.001) & (d < zfar)
        padded = _fb_np.pad(d, radius, mode='constant', constant_values=0)
        pv = _fb_np.pad(valid, radius, mode='constant', constant_values=False)
        bad = _fb_np.zeros((H, W), dtype=_fb_np.float32)
        total = 0
        for dh in range(-radius, radius + 1):
            for dw in range(-radius, radius + 1):
                nh, nw = radius + dh, radius + dw
                nb = padded[nh:nh+H, nw:nw+W]
                nv = pv[nh:nh+H, nw:nw+W]
                bad += ((~nv) | (_fb_np.abs(nb - d) > depth_diff_thres)).astype(_fb_np.float32)
                total += 1
        out = _fb_np.where((bad / total > ratio_thres) | (~valid), 0.0, d).astype(_fb_np.float32)
        return _t.from_numpy(out).to(device) if is_tensor else out

if 'bilateral_filter_depth' not in dir():
    def bilateral_filter_depth(depth, radius=2, zfar=100,
                               sigmaD=2, sigmaR=100000, device='cuda'):
        import torch as _t, cv2
        is_tensor = isinstance(depth, _t.Tensor)
        d = depth.cpu().numpy() if is_tensor else _fb_np.array(depth)
        valid = (d >= 0.001) & (d < zfar)
        out = cv2.bilateralFilter(d.astype(_fb_np.float32),
                                  d=2*radius+1, sigmaColor=float(sigmaR), sigmaSpace=float(sigmaD))
        out = _fb_np.where(valid, out, 0.0).astype(_fb_np.float32)
        return _t.from_numpy(out).to(device) if is_tensor else out
"""
    with open(_utils_path, 'a') as _f:
        _f.write(_fallback_code)
    print("  [OK] Fallbacks warp inyectados en Utils.py")
else:
    print("[OK] Fallbacks warp ya presentes en Utils.py")

# ============================================================
# DEPENDENCIAS DEL SISTEMA (C++, OpenGL, CUDA)
# ============================================================
os.environ['CUDA_HOME'] = '/usr/local/cuda'
os.environ['PATH'] = f"/usr/local/cuda/bin:{os.environ['PATH']}"

print("[1/6] Dependencias del sistema...")
!apt-get update -qq 2>&1 | tail -1
!apt-get install -q -y ninja-build \
    libgl1-mesa-dev libegl1-mesa-dev libgles2-mesa-dev libglvnd-dev \
    libboost-all-dev libeigen3-dev 2>&1 | tail -3

# ============================================================
# DEPENDENCIAS PYTHON
# ============================================================
print("\n[2/6] Dependencias Python...")
!pip install -q ninja cmake pybind11 \
    trimesh opencv-python-headless scipy scikit-learn \
    ruamel.yaml kornia transformations omegaconf einops

# ============================================================
# MOCK open3d (solo usado en debug, no en inferencia)
# ============================================================
print("\n[3/6] Mock open3d...")
import types
import numpy as _np

open3d_mock = types.ModuleType('open3d')
for sub in ['geometry', 'io', 'utility', 'visualization', 'core', 't', 'pipelines']:
    m = types.ModuleType(f'open3d.{sub}')
    setattr(open3d_mock, sub, m)
    sys.modules[f'open3d.{sub}'] = m

class _MockPC:
    def __init__(self):
        self.points = None
        self.colors = None
        self.normals = None
    def voxel_down_sample(self, voxel_size):
        if self.points is None: return self
        pts = _np.asarray(self.points)
        if len(pts) == 0: return self
        q = _np.round(pts / voxel_size).astype(_np.int32)
        _, idx = _np.unique(q, axis=0, return_index=True)
        r = _MockPC()
        r.points = open3d_mock.utility.Vector3dVector(pts[idx])
        if self.normals is not None:
            r.normals = open3d_mock.utility.Vector3dVector(_np.asarray(self.normals)[idx])
        return r
    def paint_uniform_color(self, c): return self
    def estimate_normals(self, **kw): pass
    def transform(self, T):
        if self.points is not None:
            pts = _np.asarray(self.points)
            pts_h = _np.c_[pts, _np.ones(len(pts))]
            pts_t = (T @ pts_h.T).T[:, :3]
            self.points = open3d_mock.utility.Vector3dVector(pts_t)
        return self

open3d_mock.geometry.PointCloud = _MockPC
open3d_mock.utility.Vector3dVector = lambda x: _np.asarray(x)
open3d_mock.io.write_point_cloud = lambda *a, **k: None
open3d_mock.io.read_point_cloud = lambda *a, **k: _MockPC()
sys.modules['open3d'] = open3d_mock
print("[OK] open3d mockeado (con voxel_down_sample)")

print("\n[4/6] nvdiffrast...")
try:
    import nvdiffrast
    print(f"[OK] nvdiffrast ya instalado")
except ImportError:
    !pip install nvdiffrast 2>&1 | tail -3
    try:
        import nvdiffrast
        print("[OK] nvdiffrast desde PyPI")
    except ImportError:
        print("PyPI fallo, compilando desde source...")
        !pip install --no-build-isolation git+https://github.com/NVlabs/nvdiffrast 2>&1 | tail -10

# ============================================================
# PYTORCH3D
# ============================================================
print("\n[5/6] pytorch3d...")
try:
    import pytorch3d
    print(f"[OK] pytorch3d ya instalado")
except ImportError:
    print("Compilando pytorch3d desde source (~10-30 min)...")
    !pip install git+https://github.com/facebookresearch/pytorch3d.git@stable 2>&1 | tail -5

# ============================================================
# COMPILAR EXTENSIONES C++/CUDA DE FOUNDATIONPOSE
# ============================================================
print("\n[6/6] Compilando extensiones FoundationPose...")

# mycpp (C++ con pybind11 + boost + eigen)
import glob as _glob
mycpp_dir = f"{FP_DIR}/mycpp"
mycpp_build = f"{mycpp_dir}/build"
_mycpp_sos = _glob.glob(f"{mycpp_build}/mycpp*.so")
if os.path.exists(mycpp_dir) and not _mycpp_sos:
    print("  Compilando mycpp...")
    os.makedirs(mycpp_build, exist_ok=True)
    !cd {mycpp_build} && cmake .. 2>&1 | tail -3
    !cd {mycpp_build} && make -j$(nproc) 2>&1 | tail -3
    _mycpp_sos = _glob.glob(f"{mycpp_build}/mycpp*.so")
    print(f"  [OK] mycpp ({len(_mycpp_sos)} .so)" if _mycpp_sos else "  [WARN] mycpp no produjo .so, se usara fallback Python")
elif _mycpp_sos:
    print(f"  mycpp: ya compilado ({len(_mycpp_sos)} .so)")
else:
    print("  mycpp: directorio no encontrado")

# ---- Fallback Python para mycpp.cluster_poses ----
# mycpp C++ falla en Colab, pero cluster_poses es la unica funcion usada.
# Implementamos un fallback puro en Python/NumPy.
try:
    sys.path.insert(0, mycpp_build)  # permitir import mycpp desde .so
    import mycpp as _mycpp_real
    if not hasattr(_mycpp_real, 'cluster_poses'):
        # fallback import path usado en algunos forks
        import mycpp.build.mycpp as _mycpp_real  # noqa: F811
    print("  [OK] mycpp importado")
except Exception:
    print("  [WARN] mycpp C++ no disponible, usando fallback Python")
    import types as _types
    _mycpp_mod = _types.ModuleType('mycpp')
    _mycpp_build = _types.ModuleType('mycpp.build')
    _mycpp_inner = _types.ModuleType('mycpp.build.mycpp')

    def _rotation_geodesic(R1, R2):
        cos_a = ((_np.trace(R1 @ R2.T) - 1.0) / 2.0)
        cos_a = _np.clip(cos_a, -1.0, 1.0)
        return _np.arccos(cos_a)

    def _cluster_poses(angle_diff_deg, dist_diff, poses_in, symmetry_tfs):
        """Greedy pose clustering respecting object symmetries."""
        radian_thres = angle_diff_deg / 180.0 * _np.pi
        poses_in = _np.asarray(poses_in)
        symmetry_tfs = _np.asarray(symmetry_tfs)
        if len(poses_in) == 0:
            return poses_in
        out = [poses_in[0]]
        for i in range(1, len(poses_in)):
            cur = poses_in[i]
            is_new = True
            t1 = cur[:3, 3]
            for c in out:
                t0 = c[:3, 3]
                if _np.linalg.norm(t0 - t1) >= dist_diff:
                    continue
                for tf in symmetry_tfs:
                    cur_sym = cur @ tf
                    rd = _rotation_geodesic(cur_sym[:3, :3], c[:3, :3])
                    if rd < radian_thres:
                        is_new = False
                        break
                if not is_new:
                    break
            if is_new:
                out.append(cur)
        return _np.array(out)

    _mycpp_inner.cluster_poses = _cluster_poses
    _mycpp_build.mycpp = _mycpp_inner
    _mycpp_mod.build = _mycpp_build
    sys.modules['mycpp'] = _mycpp_mod
    sys.modules['mycpp.build'] = _mycpp_build
    sys.modules['mycpp.build.mycpp'] = _mycpp_inner
    print("  [OK] mycpp fallback Python registrado")


# mycuda (CUDA extensions)
import glob
mycuda_dirs = glob.glob(f"{FP_DIR}/**/mycuda", recursive=True)
for d in mycuda_dirs:
    setup_file = os.path.join(d, 'setup.py')
    if os.path.exists(setup_file):
        print(f"  Compilando {d}...")
        !cd {d} && python setup.py build_ext --inplace 2>&1 | tail -3

sys.path.insert(0, FP_DIR)

# ============================================================
# VERIFICACION FINAL
# ============================================================
print("\n" + "=" * 50)
print("VERIFICACION DE DEPENDENCIAS")
print("=" * 50)

import numpy as np
if int(np.__version__.split('.')[0]) < 2:
    !pip install -q "numpy>=2.0"
    import importlib; importlib.reload(np)
print(f"numpy: {np.__version__}")

checks = [
    ('nvdiffrast', 'import nvdiffrast.torch'),
    ('pytorch3d', 'import pytorch3d'),
    ('kornia', 'import kornia'),
    ('transformations', 'import transformations'),
    ('trimesh', 'import trimesh'),
    ('open3d (mock)', 'import open3d'),
]
for name, cmd in checks:
    try:
        exec(cmd)
        print(f"[OK] {name}")
    except Exception as e:
        print(f"[ERROR] {name}: {e}")

# Test FP import
print("\n--- Test import FoundationPose ---")
try:
    from estimater import FoundationPose, ScorePredictor, PoseRefinePredictor
    print("[OK] FoundationPose importado correctamente")
except Exception as e:
    print(f"[ERROR] FoundationPose: {e}")
    print("Revisa los errores arriba e instala lo que falte.")

print(f"\nFoundationPose en: {FP_DIR}")


## 4. Pesos Pre-entrenados

In [ ]:
!pip install -q gdown
import gdown, shutil

# ============================================================
# Pesos pre-entrenados FoundationPose (NVIDIA oficial)
# ============================================================
# Ref: https://github.com/NVlabs/FoundationPose#pre-trained-weights
# Folder: https://drive.google.com/drive/folders/1DFezOAD0oD1BblsXVxqDsl8fj0qzB82i
#
# Tamanos esperados (medidos en instalaciones limpias):
#   Scorer  (2024-01-11-20-02-45): ~181 MB (model_best.pth)
#   Refiner (2023-10-28-18-33-37): ~65 MB  (model_best.pth)

SCORER_DIR  = f"{FP_DIR}/weights/2024-01-11-20-02-45"
REFINER_DIR = f"{FP_DIR}/bundlesdf/ckpts/2023-10-28-18-33-37"
GDRIVE_WEIGHTS = "/content/drive/MyDrive/TFM/weights/foundationpose"
os.makedirs(GDRIVE_WEIGHTS, exist_ok=True)

SCORER_MIN_MB  = 150  # real ~181 MB
REFINER_MIN_MB = 50   # real ~65 MB

def _ckpt_size_mb(d):
    if not os.path.isdir(d): return 0
    total = 0
    for root, _, files in os.walk(d):
        for fn in files:
            if fn.endswith((".pth", ".ckpt", ".bin", ".safetensors")):
                total += os.path.getsize(os.path.join(root, fn))
    return total / 1e6

def _weights_ok(path, min_mb):
    mb = _ckpt_size_mb(path)
    return mb >= min_mb, mb

def download_fp_weights():
    WEIGHTS_FOLDER_ID = "1DFezOAD0oD1BblsXVxqDsl8fj0qzB82i"

    # Paso 1: si FP_DIR ya tiene pesos validos, no toques nada.
    s_ok, s_mb = _weights_ok(SCORER_DIR, SCORER_MIN_MB)
    r_ok, r_mb = _weights_ok(REFINER_DIR, REFINER_MIN_MB)
    if s_ok and r_ok:
        print(f"[OK] Scorer {s_mb:.1f} MB / Refiner {r_mb:.1f} MB ya en {FP_DIR}")
        return True

    # Paso 2: verificar cache en Drive.
    drive_scorer  = f"{GDRIVE_WEIGHTS}/2024-01-11-20-02-45"
    drive_refiner = f"{GDRIVE_WEIGHTS}/2023-10-28-18-33-37"
    ds_ok, ds_mb = _weights_ok(drive_scorer, SCORER_MIN_MB)
    dr_ok, dr_mb = _weights_ok(drive_refiner, REFINER_MIN_MB)

    if ds_ok and dr_ok:
        print(f"Copiando pesos desde Drive (cache): Scorer {ds_mb:.1f} MB, Refiner {dr_mb:.1f} MB")
        os.makedirs(os.path.dirname(SCORER_DIR), exist_ok=True)
        os.makedirs(os.path.dirname(REFINER_DIR), exist_ok=True)
        if not os.path.exists(SCORER_DIR):  shutil.copytree(drive_scorer, SCORER_DIR)
        if not os.path.exists(REFINER_DIR): shutil.copytree(drive_refiner, REFINER_DIR)
        print("[OK] Pesos copiados desde Drive")
        return True

    # Paso 3: cache Drive invalido/parcial. Si parcial, purga antes de bajar.
    if os.path.exists(drive_scorer) and not ds_ok:
        print(f"[WARN] Drive scorer parcial ({ds_mb:.1f} MB < {SCORER_MIN_MB} MB). Purgando.")
        shutil.rmtree(drive_scorer, ignore_errors=True)
    if os.path.exists(drive_refiner) and not dr_ok:
        print(f"[WARN] Drive refiner parcial ({dr_mb:.1f} MB < {REFINER_MIN_MB} MB). Purgando.")
        shutil.rmtree(drive_refiner, ignore_errors=True)

    # Paso 4: descargar desde Google Drive oficial de NVIDIA.
    print(f"Descargando pesos desde Google Drive de NVIDIA (~500 MB)...")
    print(f"(se cachea en {GDRIVE_WEIGHTS} para futuras sesiones)")

    download_dir = f"{GDRIVE_WEIGHTS}/download_temp"
    os.makedirs(download_dir, exist_ok=True)

    try:
        gdown.download_folder(
            url=f"https://drive.google.com/drive/folders/{WEIGHTS_FOLDER_ID}",
            output=download_dir, quiet=False
        )
    except Exception as e:
        print(f"Error con gdown: {e}")
        print(f"\nDescarga manual:")
        print(f"1. https://drive.google.com/drive/folders/{WEIGHTS_FOLDER_ID}")
        print(f"2. Sube a {GDRIVE_WEIGHTS}/")
        return False

    # Mover a ubicaciones correctas (Drive + FP_DIR)
    for item in os.listdir(download_dir):
        src = os.path.join(download_dir, item)
        if "2024-01-11" in item:  # Scorer
            dst_drive, dst_fp = drive_scorer, SCORER_DIR
        elif "2023-10-28" in item:  # Refiner
            dst_drive, dst_fp = drive_refiner, REFINER_DIR
        else:
            continue
        if os.path.isdir(src) and not os.path.exists(dst_drive):
            shutil.copytree(src, dst_drive)
            print(f"  cacheado: {dst_drive}")
        os.makedirs(os.path.dirname(dst_fp), exist_ok=True)
        if not os.path.exists(dst_fp):
            shutil.copytree(src, dst_fp)
            print(f"  copiado a FP: {dst_fp}")

    shutil.rmtree(download_dir, ignore_errors=True)

    # Paso 5: validacion final.
    s_ok, s_mb = _weights_ok(SCORER_DIR, SCORER_MIN_MB)
    r_ok, r_mb = _weights_ok(REFINER_DIR, REFINER_MIN_MB)
    if not (s_ok and r_ok):
        print(f"[ERR] Post-download: Scorer {s_mb:.1f}/{SCORER_MIN_MB} MB, Refiner {r_mb:.1f}/{REFINER_MIN_MB} MB")
        return False
    return True

success = download_fp_weights()

# PoseRefinePredictor busca en weights/, no en bundlesdf/ckpts/
refiner_in_weights = f"{FP_DIR}/weights/2023-10-28-18-33-37"
if os.path.exists(REFINER_DIR) and not os.path.exists(refiner_in_weights):
    shutil.copytree(REFINER_DIR, refiner_in_weights)
    print(f"Refiner enlazado en {refiner_in_weights}")

if not success:
    raise RuntimeError(
        "Pesos FoundationPose no disponibles o corruptos. "
        "Revisa los mensajes arriba. No continues con celdas de inferencia."
    )

print("\n" + "=" * 50)
print("[OK] Pesos de FoundationPose listos")
print("=" * 50)
!ls -lh {SCORER_DIR}/ 2>/dev/null | head -5
!ls -lh {REFINER_DIR}/ 2>/dev/null | head -5


## 5. Inicializar FoundationPose

In [ ]:
import trimesh

# Imports defensivos: si la celda 6 fallo silenciosamente, avisar claro
try:
    import nvdiffrast.torch as dr
except Exception as _e:
    raise RuntimeError(
        'nvdiffrast no se importo. Re-ejecuta la celda de instalacion (seccion 3). '
        f'Error original: {_e}'
    )

sys.path.insert(0, FP_DIR)
try:
    from estimater import FoundationPose, ScorePredictor, PoseRefinePredictor
except Exception as _e:
    raise RuntimeError(
        'No se pudo importar FoundationPose. Verifica que la seccion 3 termino sin errores '
        'y que los pesos estan descargados (seccion 4). '
        f'Error original: {_e}'
    )

# --- Inicializar los modelos (una sola vez) ---
print("Cargando ScorePredictor...")
scorer = ScorePredictor()
print("[OK] ScorePredictor cargado")

print("Cargando PoseRefinePredictor...")
refiner = PoseRefinePredictor()
print("[OK] PoseRefinePredictor cargado")

print("Creando contexto de rasterizacion CUDA...")
glctx = dr.RasterizeCudaContext()
print("[OK] nvdiffrast glctx creado")

print("\nFoundationPose listo para inferencia.")


In [ ]:
# --- Cargar meshes de los objetos ---
# BOP format: data/datasets/{ycbv,tless}/models/obj_XXXXXX.ply

import json

DATA_DIR = f"{REPO_DIR}/data/datasets"

def load_bop_meshes(dataset_dir):
    """Carga todos los meshes de un dataset BOP.

    Returns:
        dict: {obj_id: trimesh.Trimesh}
    """
    models_dir = Path(dataset_dir) / 'models'
    meshes = {}

    for ply_file in sorted(models_dir.glob('obj_*.ply')):
        # obj_000001.ply -> obj_id = 1
        obj_id = int(ply_file.stem.split('_')[1])
        mesh = trimesh.load(str(ply_file), process=False)

        # BOP meshes estan en mm, convertir a metros
        mesh.vertices *= 1e-3

        # Garantizar vertex_normals para FoundationPose
        if mesh.vertex_normals is None or len(mesh.vertex_normals) == 0:
            mesh.fix_normals()

        meshes[obj_id] = mesh

    return meshes


def create_estimator(mesh, scorer, refiner, glctx):
    """Crea un estimador FoundationPose para un objeto dado."""
    est = FoundationPose(
        model_pts=mesh.vertices.copy(),
        model_normals=mesh.vertex_normals.copy(),
        mesh=mesh,
        scorer=scorer,
        refiner=refiner,
        glctx=glctx,
        debug=0,
    )
    return est


# Cargar meshes YCB-V
ycbv_meshes = load_bop_meshes(f"{DATA_DIR}/ycbv")
print(f"YCB-V: {len(ycbv_meshes)} meshes cargados (obj_ids: {sorted(ycbv_meshes.keys())})")

# Cargar meshes T-LESS
tless_path = Path(DATA_DIR) / 'tless'
if (tless_path / 'models').exists():
    tless_meshes = load_bop_meshes(f"{DATA_DIR}/tless")
    print(f"T-LESS: {len(tless_meshes)} meshes cargados")
else:
    tless_meshes = {}
    print("T-LESS: modelos no disponibles")

# --- Configuracion de evaluacion ---
# MODE controla el volumen de trabajo. Cambia solo esta linea.
#   'smoke' : 1 escena x 3 imagenes  (~2-3 min)  -> validacion rapida
#   'dev'   : 5 escenas x 50 imagenes (~30-60 min) -> iteracion
#   'full'  : todas las escenas y imagenes (HORAS, requiere checkpoints)
MODE = 'dev'  # run completo: 5 escenas x 50 imgs por dataset (~30-45 min cada uno)

_PRESETS = {
    'smoke': dict(MAX_SCENES=1, MAX_IMAGES_PER_SCENE=3,  REGISTER_ITERATIONS=5,  CKPT_EVERY_IMGS=10),
    'dev':   dict(MAX_SCENES=5, MAX_IMAGES_PER_SCENE=50, REGISTER_ITERATIONS=5,  CKPT_EVERY_IMGS=25),
    'full':  dict(MAX_SCENES=None, MAX_IMAGES_PER_SCENE=None, REGISTER_ITERATIONS=5, CKPT_EVERY_IMGS=50),
}
_p = _PRESETS[MODE]
MAX_SCENES = _p['MAX_SCENES']
MAX_IMAGES_PER_SCENE = _p['MAX_IMAGES_PER_SCENE']
REGISTER_ITERATIONS = _p['REGISTER_ITERATIONS']
CKPT_EVERY_IMGS = _p['CKPT_EVERY_IMGS']
print(f'MODE={MODE} -> MAX_SCENES={MAX_SCENES}, MAX_IMAGES_PER_SCENE={MAX_IMAGES_PER_SCENE}, CKPT cada {CKPT_EVERY_IMGS} imgs')

### Smoke test (1 inferencia antes del loop grande)

Corre UNA inferencia sobre la primera imagen de la primera escena YCB-V y valida que los rangos sean sensatos. Si esta celda falla o reporta ADD absurdo, **no continues** — algo (unidades, K, mascara, pesos) esta mal. Mejor detectarlo aqui que tras 40 minutos de inferencia.


In [ ]:
# SMOKE_TEST_V3 -- valida (a) pipeline end-to-end y (b) fix mask-per-gt_idx en obj con gt_idx>0
from src.utils.dataset_loader import BOPDataset
import numpy as _np
import cv2

print('=== SMOKE TEST ===')
_ds = BOPDataset(f'{DATA_DIR}/ycbv', split='test')
_targets = _ds.load_bop_test_targets()
assert _targets, 'test_targets_bop19.json no encontrado en YCB-V'

# Buscar un (scene, img) que tenga >=2 targets con meshes disponibles,
# y escoger como objetivo el que esta en gt_idx>0 (no el primero).
# Esto garantiza que el smoke ejerce el fix mask-per-gt_idx.
from collections import defaultdict as _dd
_by_si = _dd(list)
for _t in _targets:
    if _t['obj_id'] in ycbv_meshes:
        _by_si[(f"{_t['scene_id']:06d}", int(_t['im_id']))].append(int(_t['obj_id']))

_sid = _iid = _obj_id = _gt_idx = None
for (s, i), obj_ids in _by_si.items():
    if len(obj_ids) < 2:
        continue
    _gt_list_tmp = _ds.load_scene_gt(s).get(str(i), [])
    # Buscar un obj cuyo gt_idx > 0
    for _o in obj_ids:
        _idx = next((k for k, g in enumerate(_gt_list_tmp) if g['obj_id'] == _o), None)
        if _idx is not None and _idx > 0:
            _sid, _iid, _obj_id, _gt_idx = s, i, _o, _idx
            break
    if _sid is not None:
        break

# Fallback: si no hay (sid,iid) con obj en gt_idx>0, usar el primer target disponible
if _sid is None:
    for (s, i), obj_ids in _by_si.items():
        _gt_list_tmp = _ds.load_scene_gt(s).get(str(i), [])
        for _o in obj_ids:
            _idx = next((k for k, g in enumerate(_gt_list_tmp) if g['obj_id'] == _o), None)
            if _idx is not None:
                _sid, _iid, _obj_id, _gt_idx = s, i, _o, _idx
                break
        if _sid is not None:
            break
assert _sid is not None, 'Ningun BOP target matchea con meshes disponibles'
print(f'  Target elegido: scene={_sid}, img={_iid}, obj_id={_obj_id}, gt_idx={_gt_idx}')
if _gt_idx == 0:
    print('  [WARN] obj_id elegido esta en gt_idx=0 — el fix mask-per-gt_idx NO se ejerce en este smoke.')

# Cargar datos (no usar sample["mask"] — es solo de gt_idx=0)
_rgb = _ds.load_rgb(_sid, _iid)
_depth_raw = _ds.load_depth(_sid, _iid)
_cam = _ds.load_scene_camera(_sid)[str(_iid)]
_K = _cam['cam_K']
_depth = _depth_raw.astype(_np.float32) * _cam['depth_scale'] * 1e-3

_valid = _depth[_depth > 0.01]
_med = float(_np.median(_valid)) if _valid.size else 0.0
print(f'  depth  shape={_depth.shape} median={_med:.3f} m  range=[{_depth.min():.3f}, {_depth.max():.3f}]')
print(f'  K[0,0]={_K[0,0]:.1f} (focal en px)  K shape={_K.shape}')

assert _K.shape == (3, 3), f'cam_K shape {_K.shape} != (3,3)'
assert 0.1 < _med < 3.0, f'depth mediana {_med:.3f} m fuera de rango (0.1-3.0 m). Revisa depth_scale.'
assert 300 < _K[0,0] < 3000, f'focal {_K[0,0]:.1f} px fuera de rango. Revisa scene_camera.json'

_gt = _ds.load_scene_gt(_sid).get(str(_iid), [])[_gt_idx]
assert _gt['obj_id'] == _obj_id, f'gt_idx/obj_id inconsistente'

# Carga mask per gt_idx correcto y verifica distinta de mask gt_idx=0 si aplica
_mask = _ds.load_mask(_sid, _iid, obj_idx=_gt_idx, visible_only=True)
assert _mask is not None, f'mask_visib faltante para gt_idx={_gt_idx}'
_mask = _mask.astype(_np.uint8)
assert _mask.sum() > 100, f'Mask casi vacia ({_mask.sum()} px) — posible oclusion total'
print(f'  mask_sum(gt_idx={_gt_idx})={int(_mask.sum())} px')
if _gt_idx > 0:
    _mask0 = _ds.load_mask(_sid, _iid, obj_idx=0, visible_only=True)
    if _mask0 is not None:
        _intersect = ((_mask > 0) & (_mask0 > 0)).sum()
        _union = ((_mask > 0) | (_mask0 > 0)).sum()
        print(f'  mask_sum(gt_idx=0)={int(_mask0.sum())} px  IoU(mask, mask_gt_idx0)={_intersect/max(_union,1):.3f}')
        assert _intersect / max(_union, 1) < 0.95, (
            'Las masks de gt_idx 0 y ' + str(_gt_idx) + ' son casi iguales — sospecha de bug en load_mask'
        )

_est_smoke = create_estimator(ycbv_meshes[_obj_id], scorer, refiner, glctx)
_t0 = time.time()
_pose = _est_smoke.register(K=_K, rgb=_rgb, depth=_depth,
                            ob_mask=_mask, iteration=REGISTER_ITERATIONS)
_elapsed = time.time() - _t0
print(f'  register() OK en {_elapsed:.2f} s')

_R_pred, _t_pred = _pose[:3, :3], _pose[:3, 3] * 1000.0
_R_gt = _np.array(_gt['cam_R_m2c']).reshape(3, 3)
_t_gt = _np.array(_gt['cam_t_m2c'])
_pts_mm = _np.asarray(ycbv_meshes[_obj_id].vertices) * 1000.0
_err = _np.linalg.norm((_R_pred @ _pts_mm.T).T + _t_pred - ((_R_gt @ _pts_mm.T).T + _t_gt), axis=1).mean()
_diam = _ds.get_object_diameter(_obj_id)
print(f'  ADD={_err:.1f} mm  diametro_obj={_diam:.1f} mm  ADD/diam={_err/_diam:.2%}')

if _err > 100.0:
    print('\n[SMOKE FAIL] ADD > 100 mm en primera inferencia.')
    print('  Causas tipicas: mask de otro objeto, profundidad en mm en vez de metros,')
    print('  K equivocado, o pesos corruptos.')
    raise AssertionError(f'Smoke test fallo: ADD={_err:.1f} mm')

del _est_smoke; import gc; gc.collect(); torch.cuda.empty_cache()
print('\n[SMOKE OK] Pipeline valido (incl. fix mask-per-gt_idx si gt_idx>0).')


## 6. Inferencia YCB-Video

In [ ]:
from src.utils.dataset_loader import BOPDataset
from tqdm.notebook import tqdm
from collections import defaultdict
import cv2

# Cargar dataset YCB-V
print(f"Cargando YCB-V desde {DATA_DIR}/ycbv")
ycbv = BOPDataset(f"{DATA_DIR}/ycbv", split="test")
ycbv_scenes = ycbv.get_scene_ids()
print(f"YCB-V: {len(ycbv_scenes)} escenas de test")

# --- BOP-19 test targets (PROTOCOLO OFICIAL) ---
# YCB-V tiene ~2243 imgs/escena en rgb/, pero el eval oficial BOP mide sobre ~75/escena.
# Filtramos por test_targets_bop19.json para ser comparables al paper.
bop_targets = ycbv.load_bop_test_targets()
if not bop_targets:
    raise RuntimeError("test_targets_bop19.json no encontrado en YCB-V. Re-extrae ycbv_base.zip.")

# Agrupar: targets_by_scene[scene_id_str] -> {im_id_int: [obj_id,...]}
targets_by_scene = defaultdict(lambda: defaultdict(list))
for t in bop_targets:
    sid = f"{t['scene_id']:06d}"
    # BOP targets: inst_count indica cuantas instancias del mismo obj_id hay que evaluar
    # en esa imagen. Para YCB-V suele ser 1, para T-LESS puede ser >1. Expandimos para
    # que cada instancia ocupe un slot en target_obj_ids.
    for _ in range(int(t.get('inst_count', 1))):
        targets_by_scene[sid][int(t['im_id'])].append(int(t['obj_id']))

print(f"YCB-V BOP-19 targets: {len(bop_targets)} objetos en {len(targets_by_scene)} escenas")

eval_scenes = [s for s in ycbv_scenes if s in targets_by_scene]
eval_scenes = eval_scenes[:MAX_SCENES] if MAX_SCENES else eval_scenes
print(f"\nEvaluando {len(eval_scenes)} escenas (MAX_SCENES={MAX_SCENES})")
print(f"Iteraciones de registro: {REGISTER_ITERATIONS}")

# --- Inferencia con CHECKPOINTS ---
CHECKPOINT_DIR = "/content/drive/MyDrive/TFM/checkpoints"
CHECKPOINT_FILE = f"{CHECKPOINT_DIR}/fp_ycbv_checkpoint.json"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

results_ycbv = []
completed_scenes = set()
timing_total = 0
n_objects_evaluated = 0
n_images_evaluated = 0
errors = []

if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE) as f:
        ckpt = json.load(f)
    _prev_mode = ckpt.get("mode")
    _prev_max_imgs = ckpt.get("max_images_per_scene")
    if ckpt.get("schema_version") != "v2_bop_targets_mask_per_gt_idx":
        print(f"[WARN] Checkpoint con schema antiguo ({ckpt.get('schema_version','v1')}) - descartando.")
        print("  Las predicciones previas tienen el bug mask/target y no son recuperables.\n")
        os.remove(CHECKPOINT_FILE)
    elif _prev_mode is not None and (_prev_mode != MODE or _prev_max_imgs != MAX_IMAGES_PER_SCENE):
        print(f"[WARN] Checkpoint con MODE/limits distintos ({_prev_mode},{_prev_max_imgs} vs {MODE},{MAX_IMAGES_PER_SCENE}).")
        print("  Descartando para evitar mezclar sampleos heterogeneos.\n")
        os.remove(CHECKPOINT_FILE)
    elif ckpt.get("n_objects_evaluated", 0) > 0:
        results_ycbv = ckpt["results"]
        completed_scenes = set(ckpt["completed_scenes"])
        timing_total = ckpt["timing_total"]
        n_objects_evaluated = ckpt["n_objects_evaluated"]
        n_images_evaluated = ckpt["n_images_evaluated"]
        print(f"CHECKPOINT VALIDO (v2_bop_targets_mask_per_gt_idx): {len(completed_scenes)} escenas, {n_objects_evaluated} objetos")
        print(f"  Retomando desde escena siguiente...\n")
    else:
        print("Checkpoint anterior con 0 predicciones (descartado). Reiniciando.\n")
        os.remove(CHECKPOINT_FILE)
else:
    print("Sin checkpoint previo. Comenzando desde cero.\n")

estimator_cache = {}

for scene_idx, scene_id in enumerate(tqdm(eval_scenes, desc="Escenas YCB-V")):
    if scene_id in completed_scenes:
        continue

    scene_gt = ycbv.load_scene_gt(scene_id)
    scene_camera = ycbv.load_scene_camera(scene_id)

    # Solo imgs del BOP test subset, ordenadas y limitadas por MAX_IMAGES_PER_SCENE
    scene_target_imgs = sorted(targets_by_scene[scene_id].keys())
    image_ids = scene_target_imgs[:MAX_IMAGES_PER_SCENE]
    for img_id in image_ids:
        try:
            # Carga RGB+Depth+K una vez por imagen (mask se carga per obj_id abajo)
            rgb = ycbv.load_rgb(scene_id, img_id)
            depth_raw = ycbv.load_depth(scene_id, img_id)
            cam = scene_camera.get(str(img_id))
            if cam is None:
                continue
            cam_K = cam["cam_K"]
            depth = depth_raw.astype(np.float32) * cam["depth_scale"] * 1e-3

            img_key = str(img_id)
            gt_list = scene_gt.get(img_key, scene_gt.get(img_id, []))

            # Iterar SOLO los obj_id que BOP pide para esta (scene, img)
            target_obj_ids = targets_by_scene[scene_id][img_id]
            for target_obj_id in target_obj_ids:
                if target_obj_id not in ycbv_meshes:
                    continue

                # Buscar el gt_idx cuyo obj_id coincide con el target (BUG FIX)
                gt_idx = next((i for i, g in enumerate(gt_list)
                               if g["obj_id"] == target_obj_id), None)
                if gt_idx is None:
                    continue

                # Cargar mask especifica del gt_idx correcto (antes se reusaba mask de gt_idx=0)
                mask = ycbv.load_mask(scene_id, img_id, obj_idx=gt_idx, visible_only=True)
                if mask is None:
                    continue
                mask = mask.astype(np.uint8)

                if target_obj_id not in estimator_cache:
                    estimator_cache[target_obj_id] = create_estimator(
                        ycbv_meshes[target_obj_id], scorer, refiner, glctx
                    )
                est = estimator_cache[target_obj_id]

                t0 = time.time()
                pose_4x4 = est.register(
                    K=cam_K, rgb=rgb, depth=depth,
                    ob_mask=mask, iteration=REGISTER_ITERATIONS,
                )
                elapsed = time.time() - t0
                timing_total += elapsed
                n_objects_evaluated += 1

                R_pred = pose_4x4[:3, :3]
                t_pred = pose_4x4[:3, 3]

                results_ycbv.append({
                    "scene_id": scene_id,
                    "img_id": int(img_id),
                    "obj_id": int(target_obj_id),
                    "gt_idx": int(gt_idx),
                    "R_pred": R_pred.tolist(),
                    "t_pred": t_pred.tolist(),
                    "time_s": elapsed,
                })

            n_images_evaluated += 1

            if n_images_evaluated % CKPT_EVERY_IMGS == 0:
                with open(CHECKPOINT_FILE, 'w') as _f:
                    json.dump({
                        'results': results_ycbv,
                        'completed_scenes': list(completed_scenes),
                        'timing_total': timing_total,
                        'n_objects_evaluated': n_objects_evaluated,
                        'n_images_evaluated': n_images_evaluated,
                        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
                        'schema_version': 'v2_bop_targets_mask_per_gt_idx',
                        'mode': MODE,
                        'max_images_per_scene': MAX_IMAGES_PER_SCENE,
                    }, _f)

        except Exception as e:
            err_msg = f"Scene {scene_id} img {img_id}: {str(e)[:150]}"
            errors.append(err_msg)
            if len(errors) <= 5:
                print(f"\n[Error] {err_msg}")
            elif len(errors) % 25 == 0:
                print(f"\n[Error x{len(errors)}] ultimo: {err_msg}")
            # Abort si la tasa de error es muy alta (fallo sistemico)
            _attempted = n_objects_evaluated + len(errors)
            if _attempted >= 20 and len(errors) / _attempted > 0.30:
                print(f"\n[ABORT] Tasa de error {len(errors)}/{_attempted} > 30% — fallo sistemico.")
                print(f"  Ultimo error: {err_msg}")
                raise RuntimeError(f"Aborted YCB-V loop: {len(errors)} errores en {_attempted} intentos.")
            continue

    completed_scenes.add(scene_id)

    # --- METRICA PARCIAL por escena (solo para verificacion rapida en vivo) ---
    if results_ycbv:
        try:
            _errs_mm = []
            for _pred in results_ycbv[-200:]:
                _gt_dict = ycbv.load_scene_gt(_pred['scene_id']).get(str(_pred['img_id']), [])
                for _g in _gt_dict:
                    if _g['obj_id'] != _pred['obj_id']:
                        continue
                    _Rg = np.array(_g['cam_R_m2c']).reshape(3,3)
                    _tg = np.array(_g['cam_t_m2c'])
                    _Rp = np.array(_pred['R_pred'])
                    _tp = np.array(_pred['t_pred']) * 1000.0
                    _pts = np.asarray(ycbv_meshes[_pred['obj_id']].vertices) * 1000.0
                    _errs_mm.append(np.linalg.norm(
                        (_Rp @ _pts.T).T + _tp - ((_Rg @ _pts.T).T + _tg), axis=1).mean())
                    break
            if _errs_mm:
                print(f"  [LIVE] ADD mediano acumulado: {np.median(_errs_mm):.1f} mm (n={len(_errs_mm)})")
        except Exception as _e:
            print(f"  [LIVE] metrica parcial skip: {_e}")

    ckpt_data = {
        "results": results_ycbv,
        "completed_scenes": list(completed_scenes),
        "timing_total": timing_total,
        "n_objects_evaluated": n_objects_evaluated,
        "n_images_evaluated": n_images_evaluated,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "schema_version": "v2_bop_targets_mask_per_gt_idx",
        "mode": MODE,
        "max_images_per_scene": MAX_IMAGES_PER_SCENE,
    }
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump(ckpt_data, f)
    print(f"  [CKPT] Guardado: {len(completed_scenes)} escenas, {n_objects_evaluated} objetos")

# Checkpoint final
if results_ycbv:
    ckpt_data = {
        "results": results_ycbv,
        "completed_scenes": list(completed_scenes),
        "timing_total": timing_total,
        "n_objects_evaluated": n_objects_evaluated,
        "n_images_evaluated": n_images_evaluated,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "status": "COMPLETED",
        "schema_version": "v2_bop_targets_mask_per_gt_idx",
        "mode": MODE,
        "max_images_per_scene": MAX_IMAGES_PER_SCENE,
    }
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump(ckpt_data, f)

# --- Resumen ---
avg_time = timing_total / max(n_objects_evaluated, 1)
print(f"\n{'=' * 60}")
print(f"  FoundationPose YCB-Video -- Resumen")
print(f"{'=' * 60}")
print(f"  Imagenes evaluadas:    {n_images_evaluated}")
print(f"  Objetos evaluados:     {n_objects_evaluated}")
print(f"  Predicciones totales:  {len(results_ycbv)}")
print(f"  Tiempo promedio/obj:   {avg_time*1000:.1f} ms")
if avg_time > 0:
    print(f"  Throughput:            {1/avg_time:.1f} objetos/s")
if errors:
    print(f"  Errores:               {len(errors)}")
print(f"  Checkpoint:            {CHECKPOINT_FILE}")
print(f"{'=' * 60}")

# --- VRAM cleanup YCB-V -> T-LESS ---
import gc
estimator_cache.clear()
del estimator_cache
gc.collect()
torch.cuda.empty_cache()
_free, _total = torch.cuda.mem_get_info()
print(f"[VRAM] libre: {_free/1e9:.1f} GB / {_total/1e9:.1f} GB tras limpiar YCB-V")


## 7. Metricas YCB-Video

In [ ]:
from src.utils.metrics import add_metric, add_s_metric, compute_recall, compute_auc

if results_ycbv:
    add_errors = []
    adds_errors = []

    for pred in tqdm(results_ycbv, desc="Calculando metricas YCB-V"):
        scene_id = pred['scene_id']
        img_id = pred['img_id']
        obj_id = pred['obj_id']

        scene_gt = ycbv.load_scene_gt(scene_id)
        img_key = str(img_id)
        gt_list = scene_gt.get(img_key, scene_gt.get(img_id, []))

        # Preferir emparejamiento por gt_idx (guardado durante la inferencia).
        # Fallback: primer obj_id que matchee (compat. con checkpoints viejos).
        gt = None
        gi = pred.get('gt_idx')
        if gi is not None and gi < len(gt_list) and gt_list[gi]['obj_id'] == obj_id:
            gt = gt_list[gi]
        else:
            gt = next((g for g in gt_list if g['obj_id'] == obj_id), None)
        if gt is None:
            continue

        R_gt = np.array(gt['cam_R_m2c']).reshape(3, 3)
        t_gt = np.array(gt['cam_t_m2c']).reshape(3) * 1e-3  # mm -> m
        R_pred = np.array(pred['R_pred'])
        t_pred = np.array(pred['t_pred'])

        if obj_id in ycbv_meshes:
            pts = ycbv_meshes[obj_id].vertices  # Ya en metros
            add_errors.append(add_metric(R_pred, t_pred, R_gt, t_gt, pts))
            adds_errors.append(add_s_metric(R_pred, t_pred, R_gt, t_gt, pts))

    add_errors = np.array(add_errors)
    adds_errors = np.array(adds_errors)

    thresholds_mm = [5, 10, 20, 50]

    print(f"\n{'=' * 60}")
    print(f"  FoundationPose -- YCB-Video Metrics")
    print(f"{'=' * 60}")
    print(f"  Objetos evaluados: {len(add_errors)}")
    print(f"")
    print(f"  ADD  (mean):     {add_errors.mean()*1000:.2f} mm")
    print(f"  ADD  (median):   {np.median(add_errors)*1000:.2f} mm")
    print(f"  ADD-S (mean):    {adds_errors.mean()*1000:.2f} mm")
    print(f"  ADD-S (median):  {np.median(adds_errors)*1000:.2f} mm")
    print(f"")

    for th in thresholds_mm:
        recall_add = (add_errors * 1000 < th).mean() * 100
        recall_adds = (adds_errors * 1000 < th).mean() * 100
        print(f"  Recall@{th}mm  ADD: {recall_add:.1f}%  ADD-S: {recall_adds:.1f}%")

    auc_add = compute_auc(list(add_errors * 1000), max_threshold=100)
    auc_adds = compute_auc(list(adds_errors * 1000), max_threshold=100)
    print(f"")
    print(f"  AUC ADD:   {auc_add:.4f}")
    print(f"  AUC ADD-S: {auc_adds:.4f}")
    print(f"{'=' * 60}")

    metrics_fp_ycbv = {
        'add_mean_mm': float(add_errors.mean() * 1000),
        'add_median_mm': float(np.median(add_errors) * 1000),
        'adds_mean_mm': float(adds_errors.mean() * 1000),
        'adds_median_mm': float(np.median(adds_errors) * 1000),
        'auc_add': float(auc_add),
        'auc_adds': float(auc_adds),
        'n_evaluated': len(add_errors),
    }
    for th in thresholds_mm:
        metrics_fp_ycbv[f'recall_add_{th}mm'] = float((add_errors * 1000 < th).mean())
        metrics_fp_ycbv[f'recall_adds_{th}mm'] = float((adds_errors * 1000 < th).mean())
else:
    print("Sin predicciones YCB-V para calcular metricas.")
    metrics_fp_ycbv = {}


## 8. Inferencia T-LESS

In [ ]:
TLESS_CKPT_FILE = f"{CHECKPOINT_DIR}/fp_tless_checkpoint.json"

# Retomar desde checkpoint si existe
results_tless = []
completed_tless_scenes = set()
timing_tless = 0
n_tless_obj = 0
n_tless_images = 0
tless_errors = []

if os.path.exists(TLESS_CKPT_FILE):
    with open(TLESS_CKPT_FILE) as f:
        ckpt = json.load(f)
    _prev_mode_t = ckpt.get("mode")
    _prev_max_imgs_t = ckpt.get("max_images_per_scene")
    if ckpt.get("schema_version") != "v2_bop_targets_mask_per_gt_idx":
        print(f"[WARN] Checkpoint T-LESS con schema antiguo ({ckpt.get('schema_version','v1')}) - descartando.\n")
        os.remove(TLESS_CKPT_FILE)
    elif _prev_mode_t is not None and (_prev_mode_t != MODE or _prev_max_imgs_t != MAX_IMAGES_PER_SCENE):
        print(f"[WARN] Checkpoint T-LESS con MODE/limits distintos ({_prev_mode_t},{_prev_max_imgs_t} vs {MODE},{MAX_IMAGES_PER_SCENE}). Descartando.\n")
        os.remove(TLESS_CKPT_FILE)
    elif ckpt.get("n_objects_evaluated", 0) > 0:
        results_tless = ckpt["results"]
        completed_tless_scenes = set(ckpt["completed_scenes"])
        timing_tless = ckpt["timing_total"]
        n_tless_obj = ckpt["n_objects_evaluated"]
        n_tless_images = ckpt.get("n_images_evaluated", 0)
        print(f"CHECKPOINT T-LESS VALIDO (v2_bop_targets_mask_per_gt_idx): {len(completed_tless_scenes)} escenas, {n_tless_obj} objetos")
    else:
        print("Checkpoint T-LESS anterior con 0 predicciones (descartado). Reiniciando.\n")
        os.remove(TLESS_CKPT_FILE)
else:
    print("Sin checkpoint T-LESS previo. Comenzando desde cero.\n")

tless_test_path = Path(f"{DATA_DIR}/tless/test_primesense")

if tless_test_path.exists() and tless_meshes:
    tless = BOPDataset(f"{DATA_DIR}/tless", split="test_primesense")
    tless_scenes = tless.get_scene_ids()
    print(f"T-LESS: {len(tless_scenes)} escenas de test")

    # --- BOP-19 test targets (PROTOCOLO OFICIAL) ---
    tless_targets = tless.load_bop_test_targets()
    if not tless_targets:
        raise RuntimeError("test_targets_bop19.json no encontrado en T-LESS.")

    from collections import defaultdict
    tless_targets_by_scene = defaultdict(lambda: defaultdict(list))
    for t in tless_targets:
        sid = f"{t['scene_id']:06d}"
        # Expandir por inst_count: T-LESS tiene muchas imagenes con {inst_count: N>1} del mismo obj_id
        for _ in range(int(t.get('inst_count', 1))):
            tless_targets_by_scene[sid][int(t['im_id'])].append(int(t['obj_id']))
    print(f"T-LESS BOP-19 targets: {len(tless_targets)} objetos en {len(tless_targets_by_scene)} escenas")

    eval_tless = [s for s in tless_scenes if s in tless_targets_by_scene]
    eval_tless = eval_tless[:MAX_SCENES] if MAX_SCENES else eval_tless
    print(f"Evaluando {len(eval_tless)} escenas (MAX_SCENES={MAX_SCENES})\n")

    tless_estimator_cache = {}

    for scene_idx, scene_id in enumerate(tqdm(eval_tless, desc="Escenas T-LESS")):
        if scene_id in completed_tless_scenes:
            continue

        scene_gt = tless.load_scene_gt(scene_id)
        scene_camera = tless.load_scene_camera(scene_id)

        scene_target_imgs = sorted(tless_targets_by_scene[scene_id].keys())
        image_ids = scene_target_imgs[:MAX_IMAGES_PER_SCENE]
        for img_id in image_ids:
            try:
                rgb = tless.load_rgb(scene_id, img_id)
                depth_raw = tless.load_depth(scene_id, img_id)
                cam = scene_camera.get(str(img_id))
                if cam is None:
                    continue
                cam_K = cam["cam_K"]
                depth = depth_raw.astype(np.float32) * cam["depth_scale"] * 1e-3

                img_key = str(img_id)
                gt_list = scene_gt.get(img_key, scene_gt.get(img_id, []))

                # T-LESS: varias instancias del mismo obj_id → resolvemos por orden de aparicion.
                # Si el target pide 3x obj_id=5, tomamos los 3 primeros gt_idx cuyo obj_id==5.
                target_obj_ids = tless_targets_by_scene[scene_id][img_id]
                _seen_per_obj = defaultdict(int)
                for target_obj_id in target_obj_ids:
                    if target_obj_id not in tless_meshes:
                        continue
                    # Encontrar el n-esimo gt_idx con obj_id == target_obj_id
                    matches = [i for i, g in enumerate(gt_list) if g["obj_id"] == target_obj_id]
                    n = _seen_per_obj[target_obj_id]
                    if n >= len(matches):
                        continue
                    gt_idx = matches[n]
                    _seen_per_obj[target_obj_id] += 1

                    mask = tless.load_mask(scene_id, img_id, obj_idx=gt_idx, visible_only=True)
                    if mask is None:
                        continue
                    mask = mask.astype(np.uint8)

                    if target_obj_id not in tless_estimator_cache:
                        tless_estimator_cache[target_obj_id] = create_estimator(
                            tless_meshes[target_obj_id], scorer, refiner, glctx
                        )
                    est = tless_estimator_cache[target_obj_id]

                    t0 = time.time()
                    pose_4x4 = est.register(
                        K=cam_K, rgb=rgb, depth=depth,
                        ob_mask=mask, iteration=REGISTER_ITERATIONS,
                    )
                    elapsed = time.time() - t0
                    timing_tless += elapsed
                    n_tless_obj += 1

                    results_tless.append({
                        "scene_id": scene_id,
                        "img_id": int(img_id),
                        "obj_id": int(target_obj_id),
                        "gt_idx": int(gt_idx),
                        "R_pred": pose_4x4[:3, :3].tolist(),
                        "t_pred": pose_4x4[:3, 3].tolist(),
                        "time_s": elapsed,
                    })

                n_tless_images += 1

                if n_tless_images % CKPT_EVERY_IMGS == 0:
                    with open(TLESS_CKPT_FILE, 'w') as _f:
                        json.dump({
                            'results': results_tless,
                            'completed_scenes': list(completed_tless_scenes),
                            'timing_total': timing_tless,
                            'n_objects_evaluated': n_tless_obj,
                            'n_images_evaluated': n_tless_images,
                            'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
                            'schema_version': 'v2_bop_targets_mask_per_gt_idx',
                            'mode': MODE,
                            'max_images_per_scene': MAX_IMAGES_PER_SCENE,
                        }, _f)

            except Exception as e:
                err_msg = f"Scene {scene_id} img {img_id}: {str(e)[:150]}"
                tless_errors.append(err_msg)
                if len(tless_errors) <= 5:
                    print(f"\n[Error] {err_msg}")
                elif len(tless_errors) % 25 == 0:
                    print(f"\n[Error x{len(tless_errors)}] ultimo: {err_msg}")
                _attempted = n_tless_obj + len(tless_errors)
                if _attempted >= 20 and len(tless_errors) / _attempted > 0.30:
                    print(f"\n[ABORT] Tasa de error {len(tless_errors)}/{_attempted} > 30% — fallo sistemico.")
                    print(f"  Ultimo error: {err_msg}")
                    raise RuntimeError(f"Aborted T-LESS loop: {len(tless_errors)} errores en {_attempted} intentos.")
                continue

        completed_tless_scenes.add(scene_id)

        # Metrica parcial en vivo (con gt_idx si existe para desambiguar)
        if results_tless:
            try:
                _errs_mm = []
                for _pred in results_tless[-200:]:
                    _gt_dict = tless.load_scene_gt(_pred['scene_id']).get(str(_pred['img_id']), [])
                    _gi = _pred.get('gt_idx')
                    if _gi is not None and _gi < len(_gt_dict):
                        _g = _gt_dict[_gi]
                    else:
                        _g = next((g for g in _gt_dict if g['obj_id'] == _pred['obj_id']), None)
                    if _g is None:
                        continue
                    _Rg = np.array(_g['cam_R_m2c']).reshape(3,3)
                    _tg = np.array(_g['cam_t_m2c'])
                    _Rp = np.array(_pred['R_pred'])
                    _tp = np.array(_pred['t_pred']) * 1000.0
                    _pts = np.asarray(tless_meshes[_pred['obj_id']].vertices) * 1000.0
                    _errs_mm.append(np.linalg.norm(
                        (_Rp @ _pts.T).T + _tp - ((_Rg @ _pts.T).T + _tg), axis=1).mean())
                if _errs_mm:
                    print(f"  [LIVE] ADD mediano acumulado T-LESS: {np.median(_errs_mm):.1f} mm (n={len(_errs_mm)})")
            except Exception as _e:
                print(f"  [LIVE] metrica parcial skip: {_e}")

        ckpt_data = {
            "results": results_tless,
            "completed_scenes": list(completed_tless_scenes),
            "timing_total": timing_tless,
            "n_objects_evaluated": n_tless_obj,
            "n_images_evaluated": n_tless_images,
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "schema_version": "v2_bop_targets_mask_per_gt_idx",
            "mode": MODE,
            "max_images_per_scene": MAX_IMAGES_PER_SCENE,
        }
        with open(TLESS_CKPT_FILE, "w") as f:
            json.dump(ckpt_data, f)
        print(f"  [CKPT] {len(completed_tless_scenes)} escenas, {n_tless_obj} objetos")

    if results_tless:
        ckpt_data = {
            "results": results_tless,
            "completed_scenes": list(completed_tless_scenes),
            "timing_total": timing_tless,
            "n_objects_evaluated": n_tless_obj,
            "n_images_evaluated": n_tless_images,
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "status": "COMPLETED",
            "schema_version": "v2_bop_targets_mask_per_gt_idx",
            "mode": MODE,
            "max_images_per_scene": MAX_IMAGES_PER_SCENE,
        }
        with open(TLESS_CKPT_FILE, "w") as f:
            json.dump(ckpt_data, f)

    avg_tless = timing_tless / max(n_tless_obj, 1)
    print(f"\n{'=' * 60}")
    print(f"  FoundationPose T-LESS -- Resumen")
    print(f"{'=' * 60}")
    print(f"  Imagenes evaluadas:    {n_tless_images}")
    print(f"  Objetos evaluados:     {n_tless_obj}")
    print(f"  Predicciones totales:  {len(results_tless)}")
    print(f"  Tiempo promedio/obj:   {avg_tless*1000:.1f} ms")
    if avg_tless > 0:
        print(f"  Throughput:            {1/avg_tless:.1f} objetos/s")
    if tless_errors:
        print(f"  Errores:               {len(tless_errors)}")
    print(f"  Checkpoint:            {TLESS_CKPT_FILE}")
    print(f"{'=' * 60}")

    # --- VRAM cleanup T-LESS ---
    import gc
    tless_estimator_cache.clear()
    del tless_estimator_cache
    gc.collect()
    torch.cuda.empty_cache()
    _free, _total = torch.cuda.mem_get_info()
    print(f"[VRAM] libre: {_free/1e9:.1f} GB / {_total/1e9:.1f} GB tras limpiar T-LESS")

else:
    print("T-LESS no disponible. Verifica la descarga del dataset.")


## 9. Metricas T-LESS

In [ ]:
if results_tless:
    add_errors_tless = []
    adds_errors_tless = []

    for pred in tqdm(results_tless, desc="Calculando metricas T-LESS"):
        scene_id = pred['scene_id']
        img_id = pred['img_id']
        obj_id = pred['obj_id']

        scene_gt = tless.load_scene_gt(scene_id)
        img_key = str(img_id)
        gt_list = scene_gt.get(img_key, scene_gt.get(img_id, []))

        # T-LESS tiene INSTANCIAS MULTIPLES del mismo obj_id → emparejar por gt_idx.
        gt = None
        gi = pred.get('gt_idx')
        if gi is not None and gi < len(gt_list) and gt_list[gi]['obj_id'] == obj_id:
            gt = gt_list[gi]
        else:
            # Fallback para predicciones antiguas sin gt_idx: primer match.
            gt = next((g for g in gt_list if g['obj_id'] == obj_id), None)
        if gt is None:
            continue

        R_gt = np.array(gt['cam_R_m2c']).reshape(3, 3)
        t_gt = np.array(gt['cam_t_m2c']).reshape(3) * 1e-3
        R_pred = np.array(pred['R_pred'])
        t_pred = np.array(pred['t_pred'])

        if obj_id in tless_meshes:
            pts = tless_meshes[obj_id].vertices
            add_errors_tless.append(add_metric(R_pred, t_pred, R_gt, t_gt, pts))
            adds_errors_tless.append(add_s_metric(R_pred, t_pred, R_gt, t_gt, pts))

    add_errors_tless = np.array(add_errors_tless)
    adds_errors_tless = np.array(adds_errors_tless)

    thresholds_mm = [5, 10, 20, 50]

    print(f"\n{'=' * 60}")
    print(f"  FoundationPose -- T-LESS Metrics")
    print(f"{'=' * 60}")
    print(f"  Objetos evaluados: {len(add_errors_tless)}")
    print(f"")
    print(f"  ADD  (mean):     {add_errors_tless.mean()*1000:.2f} mm")
    print(f"  ADD  (median):   {np.median(add_errors_tless)*1000:.2f} mm")
    print(f"  ADD-S (mean):    {adds_errors_tless.mean()*1000:.2f} mm")
    print(f"  ADD-S (median):  {np.median(adds_errors_tless)*1000:.2f} mm")
    print(f"")

    for th in thresholds_mm:
        recall_add = (add_errors_tless * 1000 < th).mean() * 100
        recall_adds = (adds_errors_tless * 1000 < th).mean() * 100
        print(f"  Recall@{th}mm  ADD: {recall_add:.1f}%  ADD-S: {recall_adds:.1f}%")

    auc_add_tless = compute_auc(list(add_errors_tless * 1000), max_threshold=100)
    auc_adds_tless = compute_auc(list(adds_errors_tless * 1000), max_threshold=100)
    print(f"")
    print(f"  AUC ADD:   {auc_add_tless:.4f}")
    print(f"  AUC ADD-S: {auc_adds_tless:.4f}")
    print(f"{'=' * 60}")

    metrics_fp_tless = {
        'add_mean_mm': float(add_errors_tless.mean() * 1000),
        'add_median_mm': float(np.median(add_errors_tless) * 1000),
        'adds_mean_mm': float(adds_errors_tless.mean() * 1000),
        'adds_median_mm': float(np.median(adds_errors_tless) * 1000),
        'auc_add': float(auc_add_tless),
        'auc_adds': float(auc_adds_tless),
        'n_evaluated': len(add_errors_tless),
    }
    for th in thresholds_mm:
        metrics_fp_tless[f'recall_add_{th}mm'] = float((add_errors_tless * 1000 < th).mean())
        metrics_fp_tless[f'recall_adds_{th}mm'] = float((adds_errors_tless * 1000 < th).mean())
else:
    print("Sin predicciones T-LESS para calcular metricas.")
    metrics_fp_tless = {}


## 10. Comparativa vs GDR-Net++

In [ ]:
# --- Baselines de referencia ---
# GDR-Net++ (BOP Challenge 2022 winner)
# Fuente: Hodan et al., BOP Challenge 2022, Table 3
#   https://bop.felk.cvut.cz/leaderboards/bop-challenge-2022/

gdrnet_baselines = {
    'ycbv': {
        'method': 'GDR-Net++',
        'source': 'BOP Challenge 2022',
        'vsd_recall': 0.842,
        'mssd_recall': 0.819,
        'mspd_recall': 0.874,
        'bop_avg_recall': 0.845,
    },
    'tless': {
        'method': 'GDR-Net++',
        'source': 'BOP Challenge 2022',
        'vsd_recall': 0.736,
        'mssd_recall': 0.685,
        'mspd_recall': 0.773,
        'bop_avg_recall': 0.731,
    }
}

# FoundationPose paper results (Wen et al., CVPR 2024, Table 1)
fp_paper_baselines = {
    'ycbv': {
        'method': 'FoundationPose (paper)',
        'source': 'Wen et al., CVPR 2024',
        'vsd_recall': 0.882,
        'mssd_recall': 0.862,
        'mspd_recall': 0.907,
        'bop_avg_recall': 0.884,
    },
    'tless': {
        'method': 'FoundationPose (paper)',
        'source': 'Wen et al., CVPR 2024',
        'vsd_recall': 0.774,
        'mssd_recall': 0.725,
        'mspd_recall': 0.832,
        'bop_avg_recall': 0.777,
    }
}

# --- Tabla comparativa ---
print(f"{'=' * 80}")
print(f"  COMPARATIVA: FoundationPose vs GDR-Net++")
print(f"{'=' * 80}")
print()

header = f"{'Metodo':<30} {'VSD':>8} {'MSSD':>8} {'MSPD':>8} {'BOP Avg':>8}"
sep = '-' * len(header)

for ds_name in ['ycbv', 'tless']:
    print(f"\n  Dataset: {ds_name.upper()}")
    print(f"  {sep}")
    print(f"  {header}")
    print(f"  {sep}")

    # GDR-Net++
    g = gdrnet_baselines[ds_name]
    print(f"  {g['method']:<30} {g['vsd_recall']:>8.3f} {g['mssd_recall']:>8.3f} {g['mspd_recall']:>8.3f} {g['bop_avg_recall']:>8.3f}")

    # FP paper
    fp = fp_paper_baselines[ds_name]
    print(f"  {fp['method']:<30} {fp['vsd_recall']:>8.3f} {fp['mssd_recall']:>8.3f} {fp['mspd_recall']:>8.3f} {fp['bop_avg_recall']:>8.3f}")

    # Nuestros resultados (ADD/ADD-S como proxy)
    our_metrics = metrics_fp_ycbv if ds_name == 'ycbv' else metrics_fp_tless
    if our_metrics:
        print(f"  {'FP (ours, ADD-based)':<30} {'---':>8} {'---':>8} {'---':>8} {'---':>8}")
        print(f"    ADD AUC: {our_metrics.get('auc_add', 0):.4f}")
        print(f"    ADD-S AUC: {our_metrics.get('auc_adds', 0):.4f}")
        print(f"    N evaluated: {our_metrics.get('n_evaluated', 0)}")
    else:
        print(f"  {'FP (ours)':<30} {'N/A':>8} {'N/A':>8} {'N/A':>8} {'N/A':>8}")

    print(f"  {sep}")

print(f"\n{'=' * 80}")
print("  Nota: VSD/MSSD/MSPD de GDR-Net++ y FP (paper) son del BOP leaderboard.")
print("  Nuestros resultados usan ADD/ADD-S en un subconjunto (5 escenas x 50 imgs).")
print("  Para comparacion directa, enviar al BOP evaluation server.")
print(f"{'=' * 80}")

## 11. Figuras Capitulo 6

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 12})

FIGURES_DIR = "/content/drive/MyDrive/TFM/figures/chapter6"
os.makedirs(FIGURES_DIR, exist_ok=True)

# --- Figura 1: FP paper vs GDR-Net++ en metricas BOP (VSD/MSSD/MSPD) ---
# Baseline oficial BOP — reportado por los autores. No mezclamos con ADD/ADD-S nuestros.
datasets = ['YCB-V', 'T-LESS']
metrics_names = ['VSD', 'MSSD', 'MSPD']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax_idx, (ds_label, ds_key) in enumerate(zip(datasets, ['ycbv', 'tless'])):
    ax = axes[ax_idx]
    gdr_vals = [gdrnet_baselines[ds_key][k] for k in ('vsd_recall','mssd_recall','mspd_recall')]
    fp_vals  = [fp_paper_baselines[ds_key][k] for k in ('vsd_recall','mssd_recall','mspd_recall')]

    x = np.arange(len(metrics_names)); width = 0.35
    bars1 = ax.bar(x - width/2, [v*100 for v in gdr_vals], width,
                   label='GDR-Net++ (BOP 2022)', color='#4C72B0', edgecolor='black', linewidth=0.5)
    bars2 = ax.bar(x + width/2, [v*100 for v in fp_vals], width,
                   label='FoundationPose (CVPR 2024)', color='#DD8452', edgecolor='black', linewidth=0.5)
    for bars in (bars1, bars2):
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x()+bar.get_width()/2., h+0.5, f'{h:.1f}', ha='center', va='bottom', fontsize=9)
    ax.set_xlabel('Metrica BOP'); ax.set_ylabel('Recall (%)')
    ax.set_title(f'{ds_label}'); ax.set_xticks(x); ax.set_xticklabels(metrics_names)
    ax.set_ylim(0, 100); ax.legend(loc='lower right', fontsize=9); ax.grid(axis='y', alpha=0.3)

fig.suptitle('Comparativa BOP Metrics (baselines oficiales): FoundationPose vs GDR-Net++',
             fontsize=14, fontweight='bold')
plt.tight_layout()
fig_path = f"{FIGURES_DIR}/fp_vs_gdrnet_bop_metrics.png"
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
fig.savefig(fig_path.replace('.png', '.pdf'), bbox_inches='tight')
print(f"[Fig 1] {fig_path}")
plt.show()

# --- Figura 2: nuestros resultados reales (ADD / ADD-S AUC en subset BOP) ---
# NO mezclamos con VSD/MSSD/MSPD del paper — son metricas distintas.
# Esta figura muestra SOLO nuestros numeros para el TFM, clara sobre lo que mide.
_has_ours = (metrics_fp_ycbv and metrics_fp_ycbv.get('n_evaluated', 0) > 0) or \
            ('metrics_fp_tless' in dir() and metrics_fp_tless and metrics_fp_tless.get('n_evaluated', 0) > 0)

if _has_ours:
    fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5))
    for ax_idx, (ds_label, ds_key) in enumerate(zip(datasets, ['ycbv','tless'])):
        ax = axes2[ax_idx]
        ours = metrics_fp_ycbv if ds_key=='ycbv' else (metrics_fp_tless if 'metrics_fp_tless' in dir() else {})
        n_eval = ours.get('n_evaluated', 0) if ours else 0
        if not ours or n_eval == 0:
            ax.text(0.5, 0.5, f'Sin datos para {ds_label}\n(corre MODE=dev)',
                    ha='center', va='center', transform=ax.transAxes, fontsize=11, color='#888')
            ax.set_xticks([]); ax.set_yticks([]); ax.set_title(f'{ds_label}')
            continue

        # Recall@Xmm — mas interpretable que AUC en el TFM
        recall_names = ['R@5mm','R@10mm','R@20mm','R@50mm']
        add_recalls = [ours.get(f'recall_add_{t}mm', 0)*100 for t in [5,10,20,50]]
        adds_recalls = [ours.get(f'recall_adds_{t}mm', 0)*100 for t in [5,10,20,50]]

        x2 = np.arange(len(recall_names)); width = 0.35
        b1 = ax.bar(x2 - width/2, add_recalls, width, label=f'ADD (n={n_eval})',
                    color='#55A868', edgecolor='black', linewidth=0.5)
        b2 = ax.bar(x2 + width/2, adds_recalls, width, label='ADD-S',
                    color='#C44E52', edgecolor='black', linewidth=0.5)
        for bars in (b1, b2):
            for bar in bars:
                h = bar.get_height()
                ax.text(bar.get_x()+bar.get_width()/2., h+0.5, f'{h:.1f}',
                        ha='center', va='bottom', fontsize=9)
        ax.set_xticks(x2); ax.set_xticklabels(recall_names)
        ax.set_ylabel('Recall (%)'); ax.set_ylim(0, 100)
        ax.set_title(f'{ds_label}  |  AUC ADD={ours.get("auc_add",0):.3f}  AUC ADD-S={ours.get("auc_adds",0):.3f}')
        ax.legend(loc='upper left', fontsize=9); ax.grid(axis='y', alpha=0.3)

    fig2.suptitle('FoundationPose — resultados propios (subset BOP-19 test targets)',
                  fontsize=14, fontweight='bold')
    plt.tight_layout()
    fig2_path = f"{FIGURES_DIR}/fp_ours_recall_add.png"
    fig2.savefig(fig2_path, dpi=150, bbox_inches='tight')
    fig2.savefig(fig2_path.replace('.png', '.pdf'), bbox_inches='tight')
    print(f"[Fig 2] {fig2_path}")
    plt.show()
else:
    print("[Fig 2] skip — sin metricas propias. Corre celdas 18 y 20 (YCB-V) y 22, 24 (T-LESS).")


In [ ]:
from datetime import datetime

RESULTS_DIR = "/content/drive/MyDrive/TFM/experiments/foundationpose_eval"
os.makedirs(RESULTS_DIR, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

comparison = {
    'timestamp': timestamp,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    'config': {
        'max_scenes': MAX_SCENES,
        'max_images_per_scene': MAX_IMAGES_PER_SCENE,
        'register_iterations': REGISTER_ITERATIONS,
    },
    'baselines': {
        'gdrnet': gdrnet_baselines,
        'foundationpose_paper': fp_paper_baselines,
    },
    'our_results': {
        'ycbv': {
            'metrics': metrics_fp_ycbv,
            'n_predictions': len(results_ycbv),
            'timing_total_s': timing_total,
        },
        'tless': {
            'metrics': metrics_fp_tless if 'metrics_fp_tless' in dir() else {},
            'n_predictions': len(results_tless),
            'timing_total_s': timing_tless if 'timing_tless' in dir() else 0,
        },
    },
}

result_file = f"{RESULTS_DIR}/comparison_{timestamp}.json"
with open(result_file, 'w') as f:
    json.dump(comparison, f, indent=2)
print(f"[OK] Resultados guardados: {result_file}")

# Guardar tambien predicciones completas
if results_ycbv:
    preds_file = f"{RESULTS_DIR}/predictions_ycbv_{timestamp}.json"
    with open(preds_file, 'w') as f:
        json.dump({'predictions': results_ycbv, 'metrics': metrics_fp_ycbv}, f, indent=2)
    print(f"[OK] Predicciones YCB-V: {preds_file}")

if results_tless:
    preds_file = f"{RESULTS_DIR}/predictions_tless_{timestamp}.json"
    with open(preds_file, 'w') as f:
        json.dump({'predictions': results_tless, 'metrics': metrics_fp_tless}, f, indent=2)
    print(f"[OK] Predicciones T-LESS: {preds_file}")

print(f"\nPipeline completo. Resultados en Google Drive.")

## 12. Exportar lockfile de versiones (reproducibilidad)

Genera un pip freeze con las versiones exactas que funcionaron en este run. Descargar del Drive y committearlo al repo como `requirements.colab.lock.txt` permite que el tribunal del TFM sepa las versiones exactas sin necesidad de correr el notebook. Si cambiaste algo, re-ejecuta esta celda al final del run exitoso.


In [ ]:
# LOCKFILE_EXPORT_V1
import subprocess
from datetime import datetime

OUT = '/content/drive/MyDrive/TFM/requirements.colab.lock.txt'
freeze = subprocess.run(['pip', 'freeze'], capture_output=True, text=True).stdout

header = f'''# =============================================================
# Lockfile generado automaticamente por el notebook 01_foundationpose_eval
# Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
# Runtime: Colab T4 GPU
# =============================================================
# Para committear al repo:
#   1. Descargar este archivo desde /content/drive/MyDrive/TFM/
#   2. Reemplazar requirements.colab.lock.txt en el repo
#   3. git commit + push
# =============================================================

'''

with open(OUT, 'w') as f:
    f.write(header + freeze)

print(f'[OK] Lockfile guardado: {OUT}')
print(f'  Total paquetes: {len(freeze.splitlines())}')
print(f'\nPrimeras 20 lineas:')
for line in freeze.splitlines()[:20]:
    print(f'  {line}')


## 13. Resumen de persistencia (final)

Confirma que todo tu trabajo quedo guardado en Drive y es retomable en la proxima sesion. Ejecuta al final de cada run exitoso.

In [ ]:
# ============================================================
# Persistence summary — que quedo guardado en Drive
# ============================================================
# Este resumen sirve para:
# 1. Ti: confirmar que tus resultados estan seguros antes de cerrar Colab.
# 2. Evaluadores: localizar rapido los artefactos del TFM.
# 3. Retome: saber que ya tienes y que falta si reabres el notebook.
import os
from pathlib import Path

def _size_mb(p):
    if not os.path.exists(p): return 0
    if os.path.isfile(p): return os.path.getsize(p) / 1e6
    total = 0
    for r, _, fs in os.walk(p):
        for f in fs:
            try: total += os.path.getsize(os.path.join(r, f))
            except OSError: pass
    return total / 1e6

DRIVE_ROOT = "/content/drive/MyDrive/TFM"

print("=" * 70)
print("  PERSISTENCIA EN DRIVE — estado final")
print(f"  {DRIVE_ROOT}")
print("=" * 70)

sections = [
    ("Pesos FoundationPose", f"{DRIVE_ROOT}/weights/foundationpose", [
        ("Scorer  (2024-01-11-20-02-45)", f"{DRIVE_ROOT}/weights/foundationpose/2024-01-11-20-02-45", 150),
        ("Refiner (2023-10-28-18-33-37)", f"{DRIVE_ROOT}/weights/foundationpose/2023-10-28-18-33-37", 50),
    ]),
    ("Datasets zips (cache opcional)", f"{DRIVE_ROOT}/datasets_zips", [
        ("ycbv_base",         f"{DRIVE_ROOT}/datasets_zips/ycbv_base.zip",         0),
        ("ycbv_models",       f"{DRIVE_ROOT}/datasets_zips/ycbv_models.zip",       300),
        ("ycbv_test_all",     f"{DRIVE_ROOT}/datasets_zips/ycbv_test_all.zip",     14000),
        ("tless_base",        f"{DRIVE_ROOT}/datasets_zips/tless_base.zip",        0),
        ("tless_models",      f"{DRIVE_ROOT}/datasets_zips/tless_models.zip",      20),
        ("tless_test_prim",   f"{DRIVE_ROOT}/datasets_zips/tless_test_primesense_all.zip", 7000),
    ]),
    ("Checkpoints (retome)", f"{DRIVE_ROOT}/checkpoints", [
        ("YCB-V ckpt",  f"{DRIVE_ROOT}/checkpoints/fp_ycbv_checkpoint.json",  0),
        ("T-LESS ckpt", f"{DRIVE_ROOT}/checkpoints/fp_tless_checkpoint.json", 0),
    ]),
]

for title, _, items in sections:
    print(f"\n  [{title}]")
    for label, path, min_mb in items:
        mb = _size_mb(path)
        if not os.path.exists(path):
            status = "MISSING"
        elif min_mb > 0 and mb < min_mb:
            status = f"PARCIAL (<{min_mb} MB)"
        else:
            status = "OK"
        size_str = f"{mb:7.1f} MB" if mb >= 0.01 else f"{int(mb*1000):4d} KB"
        print(f"    {label:35s} {size_str}  {status}")

# Experimentos (JSONs de comparacion + predicciones)
exp_dir = Path(f"{DRIVE_ROOT}/experiments/foundationpose_eval")
print(f"\n  [Experimentos]")
if exp_dir.is_dir():
    exps = sorted(exp_dir.glob("*.json"))
    print(f"    {len(exps)} archivos JSON en {exp_dir}")
    for e in exps[-5:]:
        print(f"      {e.name:55s} {_size_mb(e):6.1f} MB")
else:
    print(f"    (sin experimentos guardados aun en {exp_dir})")

# Figuras
fig_dir = Path(f"{DRIVE_ROOT}/figures/chapter6")
print(f"\n  [Figuras]")
if fig_dir.is_dir():
    figs = sorted(fig_dir.glob("*"))
    print(f"    {len(figs)} archivos en {fig_dir}")
    for f in figs[-5:]:
        print(f"      {f.name:55s} {_size_mb(f):6.1f} MB")
else:
    print(f"    (sin figuras guardadas aun)")

# Lockfile reproducibilidad
lockfile = f"{DRIVE_ROOT}/requirements.colab.lock.txt"
print(f"\n  [Lockfile reproducibilidad]")
if os.path.exists(lockfile):
    with open(lockfile) as f:
        n_pkg = sum(1 for line in f if line.strip() and not line.startswith("#"))
    print(f"    requirements.colab.lock.txt  {_size_mb(lockfile):.2f} MB  {n_pkg} paquetes")
else:
    print(f"    (no generado — ejecuta celda 31)")

print()
print("=" * 70)
print("  COMO RETOMAR EN PROXIMA SESION")
print("=" * 70)
print("  1. Nueva sesion Colab T4 GPU.")
print("  2. Ejecuta celdas 1-13 (setup). Con Drive montado:")
print("     - Pesos se copian desde cache (~10 s).")
print("     - Datasets: si zips estan OK, skip rapido; si no, re-baja.")
print("     - Checkpoints: retoma automaticamente donde dejaste.")
print("  3. Celda 14: verifica MODE (smoke/dev/full).")
print("  4. Celdas 15+ corren.")
print()
print("  Para evaluadores externos (cuenta distinta):")
print("  - Clonan el repo, abren este notebook.")
print("  - En celda 6 ponen USE_DRIVE_CACHE=False (no tienen tu Drive).")
print("  - Pesos FP se bajan automaticamente de NVIDIA oficial en celda 11.")
print("  - Datasets se bajan de HuggingFace en celda 6.")
print("  - Todo el resto funciona igual.")
print("=" * 70)
